# **RAG System for the Norman Castle of Aci**

In [ ]:
# @title
from IPython.display import Image, display

# Percorso esatto della tua immagine su Drive
image_path = "/content/drive/MyDrive/nlp_project/images/norman_castle.jpg"

# Mostra l'immagine scalata
display(Image(filename=image_path, width=500)) # puoi regolare il width in pixel
print("The Norman Castle of Aci Castello (Catania, Sicily) – The core historical domain of this RAG system.")

We install the libraries required for the project.

In [1]:
!pip install openai azure-identity faiss-cpu umap-learn sentence-transformers==5.5.1 transformers==5.0.0 plotly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.7/123.7 kB 10.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transformers-5.12.0


We import the required libraries.

In [2]:
import re
import json
import os
import faiss
import numpy as np
import pandas as pd
import umap
import textwrap
import plotly.express as px
import re
import string
import torch
import time
import random
import html
import os

from tqdm.auto import tqdm
from transformers import pipeline
from IPython.display import HTML, display
from sentence_transformers import SentenceTransformer
from sentence_transformers import CrossEncoder
from pathlib import Path
from google.colab import drive
from openai import OpenAI
from collections import defaultdict, Counter
from sentence_transformers import CrossEncoder
from dotenv import load_dotenv
from google.colab import userdata
from huggingface_hub import login

We retrieve the Hugging Face token saved in Colab Secrets and use it to automatically log in to Hugging Face from the notebook.

In [3]:
HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is not None:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("Login Hugging Face riuscito")
else:
    print("HF_TOKEN non trovato nei Colab secrets")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Login Hugging Face riuscito


We mount Google Drive.

In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


The objective of the project is to implement an extractive Question Answering system able to automatically answer questions about the Castle of Aci Castello, using Paolo Muscarà’s book *Il Castello d'Aci, nella leggenda e nella storia* as the information source. The book contains approximately **20,386 words**. The system does not generate free-form answers; instead, it identifies the most suitable textual passage within the documents and extracts a short answer that is coherent with the user’s question.

The project follows a multi-stage pipeline. First, the documents are collected, cleaned, and split into chunks, namely text segments of controlled size. This subdivision makes it possible to work with more manageable information units, suitable both for retrieval and for the subsequent automatic reading phase.

Starting from the generated chunks, a Question Answering dataset is created through the OpenAI APIs available through Azure for Students. For each text segment, question-answer pairs are generated, ensuring that the answers are literally extracted from the context. The dataset is then organized in a SQuAD-like format, so that it can be used for evaluation and for possible later training or testing phases of read comprehension models.

The semantic retrieval component is responsible for retrieving, given a user question, the most relevant chunks within the document collection. To do this, each chunk is transformed into a semantic embedding using a multilingual Sentence Transformer model and indexed with FAISS, which enables efficient vector similarity search.

After retrieving the most relevant passages, the system uses a BERT-based model for extractive read comprehension. The model receives the question and the chunks selected by the retriever as input, and then identifies within the text the span that represents the most likely answer.

Finally, the system performance is evaluated using standard metrics. For the retriever, metrics such as Top-k retrieval accuracy and answer containment accuracy are considered, in order to verify whether the system retrieves the correct chunk or at least a chunk containing the answer. For the extractive Question Answering component, metrics such as Exact Match and F1 Score can be used, as commonly done in tasks based on SQuAD-style datasets.

The project therefore makes it possible to build and evaluate a complete extractive Question Answering pipeline, combining automatic dataset generation, semantic retrieval, vector indexing with FAISS, and BERT-based read comprehension.

# **Loading the book in .txt format and creating the chunks**

This cell performs chunking on the text of the book about the Castle of Aci Castello.

The input file is a single `.txt` text file, already cleaned and organized into sections through titles marked with `###`. The code reads the text, normalizes spaces, line breaks, and empty lines, splits the content into sections, and then divides each section into smaller chunks.

Each chunk has a target size of 260 words and keeps an overlap of 60 words with the next chunk, in order to reduce the risk of losing useful information between two consecutive chunks.

Unlike simple fixed-length chunking, the code tries to avoid cutting sentences in the middle. To do so, the `find_sentence_boundary` function searches for the sentence-ending point closest to the theoretical chunk limit, considering up to 80 words before or after it. This makes the chunks more readable and more suitable for the subsequent retrieval and question answering phases.

For each chunk, several metadata fields are also saved, including:
- chunk identifier;
- title of the corresponding section;
- chunk index within the section;
- initial and final position in terms of words;
- full text of the chunk.

The final output is saved in the `chunks.json` file, which will be used in the subsequent phases of the project: Question Answering dataset generation, semantic retrieval, and BERT-based read comprehension.

In [5]:
CHUNK_SIZE = 260
OVERLAP = 60
MAX_LOOKAHEAD = 80

input_path = Path("/content/drive/MyDrive/nlp_project/data/libro_completo.txt")
output_path = Path("/content/drive/MyDrive/nlp_project/data/chunks.json")

text = input_path.read_text(encoding="utf-8")

text = re.sub(r"\r\n", "\n", text)
text = re.sub(r"\n{3,}", "\n\n", text)
text = re.sub(r"[ \t]+", " ", text).strip()

sections = re.split(r"(?=^### .+$)", text, flags=re.MULTILINE)

def find_sentence_boundary(words, target_end, max_lookahead=80):
    best_end = min(target_end, len(words))

    search_start = max(0, target_end - max_lookahead)
    search_end = min(len(words), target_end + max_lookahead)

    for i in range(target_end, search_end):
        if re.search(r'[.!?…]["»”\')\]]?$', words[i]):
            return i + 1

    for i in range(target_end - 1, search_start - 1, -1):
        if re.search(r'[.!?…]["»”\')\]]?$', words[i]):
            return i + 1

    return best_end

chunks = []
chunk_id = 1

for section in sections:
    section = section.strip()

    if not section:
        continue

    lines = section.splitlines()
    title_line = lines[0].strip()
    section_title = title_line.replace("###", "").strip()
    section_text = "\n".join(lines[1:]).strip()

    words = section_text.split()

    if len(words) == 0:
        continue

    start = 0
    chunk_index_in_section = 0

    while start < len(words):
        target_end = start + CHUNK_SIZE

        if target_end >= len(words):
            end = len(words)
        else:
            end = find_sentence_boundary(words, target_end, MAX_LOOKAHEAD)

        chunk_words = words[start:end]
        chunk_text = " ".join(chunk_words)

        chunks.append({
            "chunk_id": f"chunk_{chunk_id:04d}",
            "source": "libro_completo",
            "section_title": section_title,
            "chunk_index_in_section": chunk_index_in_section,
            "start_word": start,
            "end_word": end,
            "text": chunk_text
        })

        chunk_id += 1
        chunk_index_in_section += 1

        if end >= len(words):
            break

        start = max(0, end - OVERLAP)

with output_path.open("w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"Numero di chunk creati: {len(chunks)}")
print(f"File saved to: {output_path}")

Numero di chunk creati: 103
File saved to: /content/drive/MyDrive/nlp_project/data/chunks.json


We configure the Azure OpenAI credentials.

In [6]:
env_path = "/content/drive/MyDrive/nlp_project/.env"
load_dotenv(env_path)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT")

client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)

completion = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {
            "role": "user",
            "content": "Rispondi solo con: connessione riuscita"
        }
    ],
    temperature=0
)

print(completion.choices[0].message.content)

connessione riuscita


We define the prompt used to generate QA pairs from the chunk.

In [ ]:
# @title
def build_prompt(chunk):
    return f"""
Sei un esperto nella costruzione di dataset di Question Answering estrattivo in italiano per testi storici.

Devi generare coppie domanda-risposta di alta qualità a partire dal contesto fornito.

OBIETTIVO:
Creare domande naturali, autonome e utili per un sistema di Question Answering sul Castello di Aci Castello.

REGOLE OBBLIGATORIE:
1. Genera al massimo 3 coppie domanda-risposta
2. Ogni domanda deve essere autonoma: deve essere comprensibile anche senza leggere il contesto.
3. Ogni domanda deve contenere un riferimento chiaro all'evento, al personaggio, al luogo, al periodo storico o al fenomeno di cui si parla.
4. Non usare mai espressioni come:
   - "secondo il testo"
   - "secondo quanto menzionato"
   - "nel testo"
   - "nel contesto"
   - "nel brano"
   - "nella sezione"
   - "nel passo"
   - "quanto riportato"
5. Non generare domande vaghe o troppo locali, come:
   - "Chi riebbe il Castello?"
   - "Chi era assente?"
   - "Che cosa accadde?"
   - "Dove si rifugiò?"
   - "Che cosa fece?"
   - "Come viene descritto?"
6. Preferisci domande storicamente o informativamente rilevanti.
7. Evita domande su dettagli marginali, poetici, retorici o poco utili.
8. Evita domande tratte da poesie, citazioni letterarie o frasi figurative.
9. La risposta deve essere breve, precisa e copiata esattamente dal contesto.
10. La risposta deve comparire letteralmente nel contesto.
11. Non inventare informazioni.
12. Non parafrasare la risposta.
13. Alla fine, dobbiamo ottenere un dataset di esattamente 300 domande.

TIPI DI INFORMAZIONI DA PREFERIRE:
- date di eventi storici;
- nomi di personaggi;
- popoli o dominazioni;
- luoghi;
- cause esplicite;
- conseguenze esplicite;
- battaglie;
- distruzioni;
- ricostruzioni;
- passaggi di proprietà;
- caratteristiche geologiche;
- caratteristiche architettoniche;
- quantità, misure e numeri.

CONTROLLO DI COERENZA:
- Se la domanda inizia con "Chi", la risposta deve essere una persona, un popolo, un gruppo o un'istituzione.
- Se la domanda inizia con "Dove", la risposta deve essere un luogo.
- Se la domanda inizia con "Quando", "In quale anno" o "In quale data", la risposta deve essere una data, un anno o un periodo.
- Se la domanda inizia con "Quanto", la risposta deve essere una quantità, una misura o un numero.
- Se la domanda inizia con "Come", la risposta deve indicare una modalità, una caratteristica o una condizione.
- Se la domanda inizia con "Perché", la risposta deve indicare una causa o una motivazione esplicita nel contesto.
- Se non riesci a rispettare questa coerenza, non generare quella domanda.

ESEMPI DI DOMANDE SCARSE DA EVITARE:
[
  {{
    "question": "Chi riebbe il Castello?",
    "answer": "Margherita di Lauria"
  }},
  {{
    "question": "Chi era assente quando la gente si diede a pazza fuga?",
    "answer": "Blasco d'Alagona"
  }},
  {{
    "question": "Quali fenomeni chiariscono i notevoli, lenti spostamenti nelle linee di spiaggia descritti nel testo di geologia?",
    "answer": "molti fatti della stratigrafia"
  }},
  {{
    "question": "Secondo il testo, chi occupò Catania?",
    "answer": "i Romani"
  }}
]

ESEMPI DI DOMANDE MIGLIORI:
[
  {{
    "question": "Chi riebbe il Castello dopo la pace di Caltabellotta?",
    "answer": "Margherita di Lauria"
  }},
  {{
    "question": "A chi Federico II consegnò il Castello nel 1320 dopo averlo tolto a Margherita?",
    "answer": "Blasco d'Alagona il giovane"
  }},
  {{
    "question": "Chi ordinò ai suoi soldati di mettere a sacco e fuoco l'abitato di Aci?",
    "answer": "Del Balzo"
  }},
  {{
    "question": "In quale anno i Romani occuparono Catania e la rupe di Akis?",
    "answer": "263 a. C."
  }}
]

CONTROLLO FINALE:
Prima di restituire l'output, verifica che ogni domanda:
- sia naturale in italiano;
- sia autonoma;
- non contenga riferimenti artificiali al testo o al contesto;
- non sia troppo generica;
- non riguardi dettagli secondari;
- abbia una risposta breve e letterale presente nel contesto.

FORMATO DI OUTPUT:
Restituisci solo JSON valido, senza markdown, senza spiegazioni e senza testo introduttivo.

Formato richiesto:
[
  {{
    "question": "...",
    "answer": "..."
  }}
]

Titolo della sezione:
{chunk["section_title"]}

Contesto:
{chunk["text"]}
"""

Azure call for a single chunk.

In [ ]:
def generate_qa_pairs(chunk):
    prompt = build_prompt(chunk)

    response = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {
                "role": "system",
                "content": (
                    "Sei un assistente specializzato nella generazione di dataset di Question Answering estrattivo in italiano. "
                    "Genera solo domande autonome, naturali e informative. "
                    "Non usare mai espressioni come 'secondo il testo', 'nel contesto', 'nel brano', 'nella sezione' o formule simili. "
                    "Le risposte devono essere brevi e copiate esattamente dal contesto fornito. "
                    "Rispondi esclusivamente con un oggetto JSON valido nel formato: "
                    "{\"qas\": [{\"question\": \"...\", \"answer\": \"...\"}]}"
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1,
        response_format={"type": "json_object"}
    )

    return response.choices[0].message.content.strip()

Full dataset generation.

In [ ]:
chunks_path = Path("/content/drive/MyDrive/nlp_project/data/chunks.json")
output_path = Path("/content/drive/MyDrive/nlp_project/data/qa_dataset_raw.json")

chunks = json.loads(chunks_path.read_text(encoding="utf-8"))

qa_dataset = []

for chunk in chunks:
    result = generate_qa_pairs(chunk)

    try:
        parsed = json.loads(result)
        qa_pairs = parsed.get("qas", [])
    except json.JSONDecodeError:
        continue

    if not isinstance(qa_pairs, list):
        continue

    for i, qa in enumerate(qa_pairs):
        if not isinstance(qa, dict):
            continue

        answer = qa.get("answer", "").strip()
        question = qa.get("question", "").strip()
        context = chunk["text"]

        if not question or not answer:
            continue

        if answer not in context:
            continue

        qa_dataset.append({
            "id": f'{chunk["chunk_id"]}_q{i+1}',
            "chunk_id": chunk["chunk_id"],
            "source": chunk["source"],
            "section_title": chunk["section_title"],
            "context": context,
            "question": question,
            "answer": answer,
            "answer_start": context.find(answer)
        })

with output_path.open("w", encoding="utf-8") as f:
    json.dump(qa_dataset, f, ensure_ascii=False, indent=2)

print(f"Generated questions: {len(qa_dataset)}")
print(f"File saved to: {output_path}")

The dataset is then transformed into a **SQuAD-like** format, a structure commonly used for extractive Question Answering. Each example consists of a textual context, corresponding to a chunk of the corpus, and a list of questions associated with that context. For answerable questions, the answer is represented by a span contained in the context and is identified by the answer text and its initial position, `answer_start`.

In [ ]:
input_path = Path("/content/drive/MyDrive/nlp_project/data/qa_dataset_raw.json")
output_path = Path("/content/drive/MyDrive/nlp_project/data/qa_dataset_squad_like.json")

qa_dataset = json.loads(input_path.read_text(encoding="utf-8"))

grouped = defaultdict(lambda: defaultdict(list))

valid_items = 0
skipped_items = 0

for item in qa_dataset:
    context = item.get("context", "")
    question = item.get("question", "")
    answer = item.get("answer", "")
    answer_start = item.get("answer_start", -1)
    section_title = item.get("section_title", "UNKNOWN_SECTION")
    qid = item.get("id", "")

    if not context or not question or not answer or answer_start < 0 or not qid:
        skipped_items += 1
        continue

    if context[answer_start:answer_start + len(answer)] != answer:
        skipped_items += 1
        continue

    qa = {
        "id": qid,
        "question": question,
        "answers": [
            {
                "text": answer,
                "answer_start": answer_start
            }
        ],
        "is_impossible": False
    }

    grouped[section_title][context].append(qa)
    valid_items += 1

squad_data = {
    "version": "1.0",
    "data": []
}

for section_title, contexts in grouped.items():
    section_entry = {
        "title": section_title,
        "paragraphs": []
    }

    for context, qas in contexts.items():
        paragraph = {
            "context": context,
            "qas": qas
        }

        section_entry["paragraphs"].append(paragraph)

    squad_data["data"].append(section_entry)

with output_path.open("w", encoding="utf-8") as f:
    json.dump(squad_data, f, ensure_ascii=False, indent=2)

print(f"Esempi validi convertiti: {valid_items}")
print(f"Esempi scartati: {skipped_items}")
print(f"Sezioni create: {len(squad_data['data'])}")
print(f"File saved to: {output_path}")

We display the dataset:

In [ ]:
# @title
dataset_path = "/content/drive/MyDrive/nlp_project/data/qa_dataset_squad_like.json"

with open(dataset_path, "r", encoding="utf-8") as f:
    dataset = json.load(f)

items = []

for article in dataset["data"]:
    title = article["title"]

    for paragraph in article["paragraphs"]:
        context = paragraph["context"]

        for qa in paragraph["qas"]:
            answers = [a["text"] for a in qa["answers"]]

            items.append({
                "title": title,
                "id": qa["id"],
                "question": qa["question"],
                "answers": answers,
                "context": context
            })

html_cards = ""

for item in items[:10]:
    answers_html = ""

    for answer in item["answers"]:
        answers_html += f"""
        <span style="
            display: inline-block;
            background-color: #e8f4ff;
            padding: 4px 8px;
            border-radius: 6px;
            margin: 3px;
            font-size: 14px;
        ">
            {html.escape(answer)}
        </span>
        """

    html_cards += f"""
    <div style="
        border: 1px solid #ddd;
        border-radius: 10px;
        padding: 14px;
        margin-bottom: 12px;
        background-color: #fafafa;
        font-family: Arial, sans-serif;
    ">
        <div style="font-size: 13px; color: #666; margin-bottom: 6px;">
            <b>{html.escape(item["title"])}</b> · {html.escape(item["id"])}
        </div>

        <div style="font-size: 16px; margin-bottom: 10px;">
            <b>Question:</b> {html.escape(item["question"])}
        </div>

        <div style="font-size: 15px; margin-bottom: 10px;">
            <b>Answers:</b><br>
            {answers_html}
        </div>

        <details>
            <summary>Show context</summary>
            <p style="line-height: 1.5;">{html.escape(item["context"])}</p>
        </details>
    </div>
    """

display(HTML(html_cards))

# **Semantic Embedding - Retriever**

The following code builds the vector store used by the semantic retriever of the Question Answering system.

The code loads the `chunks.json` file, which contains the chunks obtained by splitting the original text. For each chunk, the text to be indexed is built by combining the section title and the chunk content. Including the title enriches the semantic representation of the passage by also providing the model with information about the historical or thematic context in which the chunk is located.

Next, each text is transformed into a semantic embedding using the `intfloat/multilingual-e5-base` model. It is an encoder-only Transformer model used to produce textual embeddings. From an architectural perspective, the base version has **12 Transformer layers** and produces **768-dimensional** embeddings. The E5 model was trained with contrastive learning on multilingual text pairs: positive pairs, namely semantically related texts, should have embeddings that are close to each other; negative pairs should have embeddings that are farther apart. The multilingual E5 technical report describes contrastive pre-training on about 1 billion multilingual text pairs, followed by fine-tuning on labeled datasets.

Since this model is designed for retrieval tasks, the chunks are preceded by the `passage:` prefix, according to the format required by the E5 model family. The embeddings are also normalized, so that the dot product can be used as a measure equivalent to cosine similarity.

The produced embeddings are converted to `float32` format and indexed with FAISS using `IndexFlatIP`, an index based on inner product. In this way, given a question, it will be possible to quickly compare its embedding with those of the chunks and retrieve the most similar passages.

In addition to the FAISS index, the code also saves a metadata file in JSON format. This file preserves the association between each indexed vector and the original chunk information, such as identifier, source, section title, and text. The metadata will be used during retrieval to map the numerical result returned by FAISS back to the actual textual content.

The final output of the cell therefore consists of two elements: the FAISS index, containing the chunk embeddings, and the metadata file, which is needed to interpret the semantic search results.

In [7]:
chunks_path = Path("/content/drive/MyDrive/nlp_project/data/chunks.json")
index_path = Path("/content/drive/MyDrive/nlp_project/data/faiss_chunks_e5.index")
metadata_path = Path("/content/drive/MyDrive/nlp_project/data/faiss_chunks_metadata_e5.json")

chunks = json.loads(chunks_path.read_text(encoding="utf-8"))

texts = [
    f"Titolo sezione: {chunk['section_title']}\nTesto: {chunk['text']}"
    for chunk in chunks
]

passages = [
    "passage: " + text
    for text in texts
]

metadata = [
    {
        "chunk_id": chunk["chunk_id"],
        "source": chunk["source"],
        "section_title": chunk["section_title"],
        "text": chunk["text"],
        "embedding_text": texts[i]
    }
    for i, chunk in enumerate(chunks)
]

model = SentenceTransformer("intfloat/multilingual-e5-base")

embeddings = model.encode(
    passages,
    convert_to_numpy=True,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

faiss.write_index(index, str(index_path))

with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Indexed chunks: {index.ntotal}")
print(f"Dimensione embedding: {dimension}")
print(f"Index saved to: {index_path}")
print(f"Metadati salvati in: {metadata_path}")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Indexed chunks: 103
Dimensione embedding: 768
Index saved to: /content/drive/MyDrive/nlp_project/data/faiss_chunks_e5.index
Metadati salvati in: /content/drive/MyDrive/nlp_project/data/faiss_chunks_metadata_e5.json


The next cell uses the chunk embeddings generated in the previous phase and projects them into a two-dimensional space using UMAP. Since the original embeddings are high-dimensional, reducing them to two dimensions makes it possible to graphically represent the semantic distribution of the chunks.

Each point in the chart represents a document chunk, while the color indicates the thematic section it belongs to. The visualization makes it possible to observe whether chunks belonging to the same section, or to semantically similar sections, tend to be positioned close to each other in space.

The chart is interactive: by hovering over a point with the mouse, it is possible to view the chunk identifier, the corresponding section, and a preview of the text. Finally, the visualization is saved in HTML format, so that it can also be consulted outside the notebook.

In [8]:
output_html = Path("/content/drive/MyDrive/nlp_project/results/embedding_chunks_umap.html")
output_html.parent.mkdir(parents=True, exist_ok=True)

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=10,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

embeddings_2d = reducer.fit_transform(embeddings)
embeddings_2d[:, 0] = -embeddings_2d[:, 0]

def wrap_hover_text(text, width=70, max_chars=450):
    text = " ".join(text.split())
    text = text[:max_chars]
    return "<br>".join(textwrap.wrap(text, width=width))

plot_data = []

for i, chunk in enumerate(chunks):
    plot_data.append({
        "x": embeddings_2d[i, 0],
        "y": embeddings_2d[i, 1],
        "chunk_id": chunk["chunk_id"],
        "section_title": chunk["section_title"],
        "preview": wrap_hover_text(chunk["text"])
    })

df = pd.DataFrame(plot_data)

section_titles = df["section_title"].drop_duplicates().tolist()

color_palette = (
    px.colors.qualitative.Alphabet +
    px.colors.qualitative.Dark24 +
    px.colors.qualitative.Light24 +
    px.colors.qualitative.Set3
)

color_map = {
    section: color_palette[i % len(color_palette)]
    for i, section in enumerate(section_titles)
}

fig = px.scatter(
    df,
    x="x",
    y="y",
    color="section_title",
    color_discrete_map=color_map,
    custom_data=["chunk_id", "section_title", "preview"],
    title="2D Visualization of Chunk Semantic Embeddings"
)

fig.update_traces(
    marker=dict(
        size=9,
        opacity=0.9,
        line=dict(
            width=0.7,
            color="black"
        )
    ),
    hovertemplate=(
        "<b>Chunk:</b> %{customdata[0]}<br>"
        "<b>Section:</b> %{customdata[1]}<br><br>"
        "<b>Preview:</b><br>%{customdata[2]}"
        "<extra></extra>"
    )
)

fig.update_layout(
    width=1400,
    height=800,
    hoverlabel=dict(
        bgcolor="white",
        font_size=12,
        font_family="Arial",
        align="left"
    ),
    legend=dict(
        title="Section",
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.02,
        font=dict(size=11)
    ),
    margin=dict(
        l=60,
        r=380,
        t=80,
        b=60
    ),
    xaxis_title="UMAP dimension 1",
    yaxis_title="UMAP dimension 2"
)

fig.write_html(
    str(output_html),
    include_plotlyjs="cdn",
    full_html=True
)

print(f"Chart saved to: {output_html}")
display(HTML(fig.to_html(include_plotlyjs="cdn")))

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Chart saved to: /content/drive/MyDrive/nlp_project/results/embedding_chunks_umap.html


To understand the effectiveness of the selected embedding model (`intfloat/multilingual-e5-base`) and validate the quality of the adopted chunking strategy (`section title + chunk text`), a visual analysis of the latent vector space was conducted. Given the high dimensionality of the vectors generated by the model, it was necessary to apply a non-linear dimensionality reduction technique. For this purpose, the **UMAP (Uniform Manifold Approximation and Projection)** algorithm was used to project the vectors into a two-dimensional space, preserving as much as possible both the local and global structure of the semantic relationships. The result of this projection is shown in Figure X.

The analysis of the spatial distribution of the chunks within the two-dimensional graph (Figure X) highlights several points of strong scientific and interpretive interest:
- **Semantic Separation and Cluster Density**: The model demonstrates excellent thematic discrimination. Highly specific sections, such as "LA PACE DI CALTABELLOTTA" or the modules dedicated to sacred architecture ("LA CHIESA DI S. MAURO", "LA CHIESA DI ACI-TREZZA"), condense into very dense micro-clusters that are clearly separated from the rest of the corpus. This maps a low internal lexical ambiguity within those chapters.
- **Chronological-Linear Correspondence** (the X-axis as a timeline): The most relevant phenomenon lies in the spontaneous organization of the horizontal axis ($UMAP\ dimension\ 1$). Although the embedding algorithm and the subsequent dimensionality reduction did not receive any explicit temporal information or date metadata, the X-axis reflects the historical progression of the text almost perfectly. Moving from left to right, one can observe a coherent transition from the geomorphological origins of the territory ("LA RUPE", "POCHE NOTE DI GEOLOGIA"), through the classical and medieval periods ("PERIODO ROMANO", "PERIODO SARACENO", "PERIODO NORMANNO"), up to the late-medieval and modern periods on the right side of the graph ("GLI ALAGONA", "RE MARTINO", "I RE DI CASTIGLIA").
- **Narrative Continuity and Transition Areas**: The presence of partially overlapping regions or vector "bridges" (as in the case of the section "IL CATACLISMA") highlights the model’s ability to capture linguistic co-occurrence and narrative continuity. These nodes represent transversal events that intersect multiple historical periods or villages, semantically linking chronologically adjacent chapters.

**Retrieval function with FAISS**

This function receives a question as input and returns the most relevant chunks retrieved through FAISS. The question is first transformed into a semantic embedding using the same model used to index the chunks, namely `intfloat/multilingual-e5-base`. Since the E5 model distinguishes between queries and documents, the `query:` prefix is added to the question.

The question embedding is normalized and converted to `float32` format, so that it can be compared with the embeddings already stored in the FAISS index. The `index.search()` function computes the similarity between the question and all indexed chunks, returning the indices of the `top_k` most similar chunks and the corresponding similarity scores.

For each retrieved result, the function uses the index returned by FAISS to access the metadata of the corresponding chunk. It then returns the chunk identifier, the section title, the similarity score, and the context text.

In this way, given a user question, the system automatically retrieves the most relevant passages of the document, which can then be used in the subsequent read comprehension phase.

In [9]:
def retrieve_faiss(question, top_k=3):
    question_embedding = model.encode(
        ["query: " + question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(question_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        item = metadata[idx]

        results.append({
            "chunk_id": item["chunk_id"],
            "section_title": item["section_title"],
            "score": float(score),
            "context": item["text"]
        })

    return results

**Retriever test**

In [10]:
question = "Chi ordinò la distruzione del castello nel 902 d.C.?"

results = retrieve_faiss(question, top_k=3)

for result in results:
    print("Chunk:", result["chunk_id"])
    print("Section:", result["section_title"])
    print("Score:", result["score"])
    print(result["context"])
    print("-" * 100)

Chunk: chunk_0021
Section: PERIODO ROMANO
Score: 0.8138833045959473
rocca e a stabilirvi un presidio militare, dando così vita alla prima fortezza. Ciò accadeva l'anno 535 d. C. (3). Osservando quello che oggi resta del Castello senza dubbio noi ci troviamo dinanzi ad una costruzione prettamente normanna e di questo ce ne dà sicurezza Pirro nel dirci che la costruzione, o meglio la sua ultima ricostruzione risale all'anno 1092. Ultima ricostruzione nelle linee generali, poiché il vecchio maniero fu costruito, rimaneggiato, rinsaldato e ampliato più volte prima e dopo il 1092. Per quanto riguarda l'epoca anteriore al 1092 alcuni storici ci narrano che la rocca fu espugnata dai Musulmani il 17 luglio del 902 d. C. (4). Questa devesi considerare l'origine storica del Castello, come opera militare, ma siccome il popolo non si contenta di quello che potrebbe essere storicamente esatto e vuole ad ogni costo risalire ancora alle origini più oscure ed impenetrabili, ecco farsi strada la leggen

**Top-k retrieval accuracy evaluation**

This cell evaluates the performance of the semantic retriever using the Question Answering dataset in SQuAD-like format.

For each question in the dataset, the correct `chunk_id` is recovered from the question identifier. Then, the `retrieve_faiss` function is used to retrieve the first 5 chunks most similar to the question according to the FAISS index.

The evaluation compares the correct chunk with the retrieved chunks and computes three metrics:

- Top-1 retrieval accuracy: checks whether the correct chunk is the first retrieved result;
- Top-3 retrieval accuracy: checks whether the correct chunk appears among the first 3 results;
- Top-5 retrieval accuracy: checks whether the correct chunk appears among the first 5 results.

The cell also saves in a list the Top-5 errors, namely the cases in which the correct chunk is not retrieved among the first 5 results. This list can later be used to qualitatively analyze retriever errors.

The goal of the evaluation is to measure how well the retrieval system can recover the correct context before the BERT read comprehension phase.

In [11]:
qa_path = Path("/content/drive/MyDrive/nlp_project/data/qa_dataset_squad_like.json")

data = json.loads(qa_path.read_text(encoding="utf-8"))

def extract_chunk_id(qid):
    return qid.rsplit("_q", 1)[0]

chunk_by_id = {
    item["chunk_id"]: item
    for item in metadata
}

total_questions = 0

top1_correct = 0
top3_correct = 0
top5_correct = 0

errors = []

for section in data["data"]:
    for paragraph in section["paragraphs"]:
        for qa in paragraph["qas"]:
            question = qa["question"]
            qid = qa["id"]
            gold_chunk_id = extract_chunk_id(qid)

            results = retrieve_faiss(question, top_k=5)
            retrieved_chunk_ids = [result["chunk_id"] for result in results]

            total_questions += 1

            if gold_chunk_id in retrieved_chunk_ids[:1]:
                top1_correct += 1

            if gold_chunk_id in retrieved_chunk_ids[:3]:
                top3_correct += 1

            if gold_chunk_id in retrieved_chunk_ids[:5]:
                top5_correct += 1

            if gold_chunk_id not in retrieved_chunk_ids[:5]:
                errors.append({
                    "id": qid,
                    "question": question,
                    "answer": qa["answers"][0]["text"],
                    "gold_chunk_id": gold_chunk_id,
                    "gold_section_title": section["title"],
                    "gold_context": chunk_by_id[gold_chunk_id]["text"],
                    "retrieved_chunks": results
                })

top1_accuracy = top1_correct / total_questions
top3_accuracy = top3_correct / total_questions
top5_accuracy = top5_correct / total_questions

print(f"Numero totale domande: {total_questions}")
print(f"Top-1 retrieval accuracy: {top1_accuracy:.4f}")
print(f"Top-3 retrieval accuracy: {top3_accuracy:.4f}")
print(f"Top-5 retrieval accuracy: {top5_accuracy:.4f}")
print(f"Errori Top-5: {len(errors)}")

Numero totale domande: 300
Top-1 retrieval accuracy: 0.6600
Top-3 retrieval accuracy: 0.8767
Top-5 retrieval accuracy: 0.9167
Errori Top-5: 25


The next cell saves the list of Top-5 errors produced during retriever evaluation in JSON format. The `errors` variable contains the cases in which, for a given question, the correct chunk is not present among the first 5 chunks retrieved by FAISS.

The file is saved in the `results` folder with the name `retrieval_top5_errors.json`. If the folder does not exist, it is created automatically. Saving the errors makes it possible to perform a later qualitative analysis, observing which questions are not handled correctly by the retriever and which chunks are retrieved instead.

In [12]:
errors_path = Path("/content/drive/MyDrive/nlp_project/results/retrieval_top5_errors.json")
errors_path.parent.mkdir(parents=True, exist_ok=True)

with errors_path.open("w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=2)

print(f"Errori Top-5 salvati: {len(errors)}")
print(f"File salvato in: {errors_path}")

Errori Top-5 salvati: 25
File salvato in: /content/drive/MyDrive/nlp_project/results/retrieval_top5_errors.json


In [13]:
errors_preview = []

for error in errors:
    retrieved_info = []

    for chunk in error["retrieved_chunks"]:
        retrieved_info.append(
            f'{chunk["chunk_id"]} | {chunk["section_title"]} | score={chunk["score"]:.4f}'
        )

    errors_preview.append({
        "id": error["id"],
        "question": error["question"],
        "answer": error["answer"],
        "gold_chunk_id": error["gold_chunk_id"],
        "gold_section_title": error["gold_section_title"],
        "retrieved_chunks": "\n".join(retrieved_info)
    })

errors_df = pd.DataFrame(errors_preview)

display(errors_df)

,id,question,answer,gold_chunk_id,gold_section_title,retrieved_chunks
0,chunk_0001_q2,Come viene chiamata localmente la piattaforma ...,Praca,chunk_0001,LA RUPE,chunk_0043 | I VESPRI SICILIANI | score=0.7977...
1,chunk_0004_q1,A quale epoca appartengono le conchiglie studi...,quaternaria,chunk_0004,LA RUPE,chunk_0003 | LA RUPE | score=0.8459\nchunk_000...
2,chunk_0006_q1,Nei versi del componimento 'Castello d'Aci' qu...,maschio tuo profilo avventuroso,chunk_0006,LA RUPE,chunk_0005 | LA RUPE | score=0.8625\nchunk_000...
3,chunk_0006_q2,"Nel componimento 'Castello d'Aci', da dove sor...",sponda cupa del mare,chunk_0006,LA RUPE,chunk_0092 | IL CANTO DEL CARCERATO | score=0....
4,chunk_0010_q3,Quale grotta viene indicata come una delle abi...,grotta che si apre sul lato nord-est,chunk_0010,I PRIMI ABITATORI DELLA TERRA D'ACI,chunk_0011 | I PRIMI ABITATORI DELLA TERRA D'A...
5,chunk_0014_q2,In quale anno il Castello sarebbe stato edific...,1200 a.C.,chunk_0014,PRIME GUERRE NEL TERRITORIO D'ACI,chunk_0020 | PERIODO ROMANO | score=0.8541\nch...
6,chunk_0027_q1,Quanto tempo durò il dominio arabo sul Castell...,170 anni,chunk_0027,PERIODO NORMANNO,chunk_0025 | PERIODO SARACENO | score=0.8536\n...
7,chunk_0035_q1,In quale anno il Castello di Aci Castello venn...,1081,chunk_0035,VALVERDE,chunk_0025 | PERIODO SARACENO | score=0.8573\n...
8,chunk_0035_q2,Chi rioccupò il Castello di Aci Castello assie...,Ruggiero,chunk_0035,VALVERDE,chunk_0103 | IL PLEBISCITO E L'UNITA' D'ITALIA...
9,chunk_0035_q3,Quale città si era ribellata ai Normanni a cau...,Catania,chunk_0035,VALVERDE,chunk_0024 | PERIODO SARACENO | score=0.8015\n...


During the qualitative error analysis, it can be observed that in some cases the gold chunk does not appear among the first 5 results retrieved by the retriever, while the immediately following or preceding chunk is retrieved instead. This behavior can be explained by the presence of overlap between consecutive chunks and by the narrative continuity of the original text. In fact, information located at the end of one chunk can be partially repeated or developed in the following chunk.

As a result, retrieving a chunk close to the gold chunk is not necessarily a substantial error, since the retrieved context may still contain useful information to answer the question. **This highlights a limitation of Top-k retrieval accuracy based exclusively on the chunk identified as gold: the metric considers only the retrieval of the same chunk from which the question was generated as correct, even when adjacent chunks contain the answer or an equivalent context.** For this reason, answer containment accuracy was also computed, checking whether the correct answer is present in the retrieved chunks.

Based on the previous observation, the following code cell checks whether, in cases where the gold chunk is not in the top-5 results, the retrieved chunk may still contain the answer. This is important because, as already mentioned, with overlap between chunks the system may retrieve a valid chunk that is different from the one labeled as gold.

In [14]:
answer_top1_correct = 0
answer_top3_correct = 0
answer_top5_correct = 0
total_questions = 0

for section in data["data"]:
    for paragraph in section["paragraphs"]:
        for qa in paragraph["qas"]:
            question = qa["question"]
            gold_answer = qa["answers"][0]["text"]

            results = retrieve_faiss(question, top_k=5)

            total_questions += 1

            contexts_top1 = [result["context"] for result in results[:1]]
            contexts_top3 = [result["context"] for result in results[:3]]
            contexts_top5 = [result["context"] for result in results[:5]]

            if any(gold_answer in context for context in contexts_top1):
                answer_top1_correct += 1

            if any(gold_answer in context for context in contexts_top3):
                answer_top3_correct += 1

            if any(gold_answer in context for context in contexts_top5):
                answer_top5_correct += 1

print(f"Answer containment Top-1: {answer_top1_correct / total_questions:.4f}")
print(f"Answer containment Top-3: {answer_top3_correct / total_questions:.4f}")
print(f"Answer containment Top-5: {answer_top5_correct / total_questions:.4f}")

Answer containment Top-1: 0.8167
Answer containment Top-3: 0.9300
Answer containment Top-5: 0.9567


This means that in 95.67% of cases, the correct answer still appears among the first 5 retrieved chunks, even if it does not always coincide with the original gold chunk. A summary of the results is shown below:

- **Embedding model**: intfloat/multilingual-e5-base
- **Indexed text**: section title + chunk text
- **Similarity**: cosine similarity via normalized embeddings + FAISS IndexFlatIP

**Retrieval accuracy**:
- **Top-1**: 0.6600
- **Top-3**: 0.8767
- **Top-5**: 0.9167

**Answer containment accuracy:**
- **Top-1**: 0.8167
- **Top-3**: 0.9300
- **Top-5**: 0.9567

# **Reranking retriever results**

The FAISS-based retriever makes it possible to quickly retrieve the chunks most similar to the question using embedding similarity. This approach is efficient and suitable for large document collections, but it has a limitation: vector similarity can retrieve chunks that are semantically close to the question but do not necessarily contain the correct answer.

In the historical domain considered here, many chunks share similar vocabulary and concepts, such as wars, sieges, reconstructions, dominations, historical figures, and dates. As a result, the retriever can retrieve passages that are thematically relevant but refer to different events.

To improve the ordering of the retrieved chunks, a reranking module is introduced. The reranker does not replace FAISS; instead, it operates on the candidates already retrieved by the retriever. In particular, FAISS is used to quickly select a broader set of candidate chunks, while the reranker evaluates more accurately the relevance of each question-context pair.

Unlike the bi-encoder retriever, which compares the question embedding and the chunk embedding separately, the reranker uses a cross-encoder model that receives the question and the context as input at the same time. In this way, it can directly model the interactions between the terms of the question and those of the textual passage, producing a more precise relevance score.

The pipeline therefore becomes:

1. FAISS retrieves the first `faiss_top_k` candidate chunks.
2. The reranker assigns a relevance score to each question-chunk pair.
3. The chunks are reordered according to the reranker score.
4. The BERT read comprehension model is applied only to the best `reader_top_k` reranked chunks.

The goal is to provide the reader with more relevant contexts, reducing the risk that BERT extracts a plausible answer from an incorrect chunk.

The model used is `mmarco-mMiniLMv2-L12-H384-v1`, a multilingual cross-encoder model. Unlike a bi-encoder, it does not separately encode question and document; instead, it receives the question-chunk pair as input and directly produces a relevance score. The model is based on multilingual MiniLMv2, with 12 Transformer layers and hidden size 384, and was trained on MMARCO, a multilingual translated version of MS MARCO.

We then load the reranker:

In [15]:
reranker = CrossEncoder(
    "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Reranker loaded")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Reranker loaded
Device: cuda


The `retrieve_faiss_reranked` function performs retrieval in two stages. In the first stage, `retrieve_faiss` retrieves a larger number of candidate chunks. In the second stage, the reranker evaluates each pair formed by the question and a retrieved chunk. The results are then sorted according to the score produced by the reranker, and only the best chunks are returned. These chunks will be used in the read comprehension phase.

In [16]:
def retrieve_faiss_reranked(question, faiss_top_k=20, rerank_top_k=5):
    initial_results = retrieve_faiss(question, top_k=faiss_top_k)

    pairs = [
        [question, result["context"]]
        for result in initial_results
    ]

    reranker_logits = reranker.predict(pairs)

    reranker_scores = 1 / (1 + np.exp(-reranker_logits))

    reranked_results = []
    for result, reranker_score in zip(initial_results, reranker_scores):
        new_result = result.copy()
        new_result["reranker_score"] = float(reranker_score)
        reranked_results.append(new_result)

    reranked_results = sorted(
        reranked_results,
        key=lambda x: x["reranker_score"],
        reverse=True
    )

    return reranked_results[:rerank_top_k]

We test the reranker on an example question, comparing the order of the chunks retrieved by FAISS with the order obtained after reranking.

In [17]:
def print_results_with_context(results, title, max_chars=900):
    print(title)

    for result in results:
        context = " ".join(result["context"].split())

        if len(context) > max_chars:
            context = context[:max_chars] + "..."

        print("-" * 100)
        print("Chunk:", result["chunk_id"])
        print("Section:", result["section_title"])
        print("FAISS score:", result["score"])

        if "reranker_score" in result:
            print("Reranker score:", result["reranker_score"])

        print("\nContext:")
        print(context)

    print("-" * 100)

In [18]:
question = "Chi ordinò la distruzione totale del Castello e volle che le pietre fossero buttate in mare?"

faiss_results = retrieve_faiss(question, top_k=5)
reranked_results = retrieve_faiss_reranked(question, faiss_top_k=20, rerank_top_k=5)

print_results_with_context(
    faiss_results,
    "FAISS results",
    max_chars=900
)

print_results_with_context(
    reranked_results,
    "Results after reranking",
    max_chars=900
)

FAISS results
----------------------------------------------------------------------------------------------------
Chunk: chunk_0045
Section: LA PACE DI CALTABELLOTTA
FAISS score: 0.8119505643844604

Context:
Nel 1302 fu conclusa la pace di Caltabellotta fra Carlo II e Federico II. In base ai trattati di quella pace il Castello fu ridato a Ruggiero di Lauria, che frattanto si era pentito del tradimento alla casa d'Aragona. Come conferma di tale pentimento, gli fu imposto di portarsi a Catania ed inginocchiarsi ai piedi di Federico II, che in quel periodo soggiornava in quella città. Nel 1305 l'ammiraglio Ruggiero di Lauria morì ed a lui successe la figlia Margherita che divenne Signora del Castello e dell'intero territorio d'Aci. Questa successione fu contestata di fatto dal Re Roberto d'Angiò in quanto, considerato che in origine il Conte Ruggiero aveva dato il Castello alla Chiesa, questa ne doveva riacquistare il dominio. Però d'altro avviso era il Re Federico II che vantava il domi

After introducing the reranking module, the retrieval evaluation is repeated using the same metrics computed for the simple FAISS retriever.

In this case, however, the evaluation is not performed directly on the original order of the chunks returned by FAISS. For each question, FAISS initially retrieves a broader set of candidates, equal to `faiss_top_k`. Then, the reranker recomputes a relevance score for each question-chunk pair and reorders the results. The Top-k metrics are therefore computed on the first chunks after reranking.

The `Top-1 retrieval accuracy` checks whether the gold chunk is in first position after reranking. The `Top-3 retrieval accuracy` checks whether the gold chunk appears among the first three reranked results. The `Top-5 retrieval accuracy` checks whether the gold chunk appears among the first five reranked results.

This evaluation makes it possible to understand whether the reranker actually improves the ordering of the retrieved chunks, and not only the final reader metrics. If the Top-k metrics increase compared to simple FAISS, this means that reranking more often brings the correct context into the first positions, providing BERT with a more suitable input for answer extraction.

In [20]:
qa_path = Path("/content/drive/MyDrive/nlp_project/data/qa_dataset_squad_like.json")

data = json.loads(qa_path.read_text(encoding="utf-8"))

def extract_chunk_id(qid):
    return qid.rsplit("_q", 1)[0]

chunk_by_id = {
    item["chunk_id"]: item
    for item in metadata
}

total_questions = 0

top1_correct = 0
top3_correct = 0
top5_correct = 0

errors_reranked = []

for section in data["data"]:
    for paragraph in section["paragraphs"]:
        for qa in paragraph["qas"]:
            question = qa["question"]
            qid = qa["id"]
            gold_chunk_id = extract_chunk_id(qid)
            gold_answers = [answer["text"] for answer in qa["answers"]]

            results = retrieve_faiss_reranked(
                question,
                faiss_top_k=20,
                rerank_top_k=5
            )

            retrieved_chunk_ids = [result["chunk_id"] for result in results]

            total_questions += 1

            if gold_chunk_id in retrieved_chunk_ids[:1]:
                top1_correct += 1

            if gold_chunk_id in retrieved_chunk_ids[:3]:
                top3_correct += 1

            if gold_chunk_id in retrieved_chunk_ids[:5]:
                top5_correct += 1

            if gold_chunk_id not in retrieved_chunk_ids[:5]:
                errors_reranked.append({
                    "id": qid,
                    "question": question,
                    "answers": gold_answers,
                    "gold_chunk_id": gold_chunk_id,
                    "gold_section_title": section["title"],
                    "gold_context": chunk_by_id[gold_chunk_id]["text"],
                    "retrieved_chunks": results
                })

top1_accuracy = top1_correct / total_questions
top3_accuracy = top3_correct / total_questions
top5_accuracy = top5_correct / total_questions

print(f"Numero totale domande: {total_questions}")
print(f"Top-1 retrieval accuracy con reranking: {top1_accuracy:.4f}")
print(f"Top-3 retrieval accuracy con reranking: {top3_accuracy:.4f}")
print(f"Top-5 retrieval accuracy con reranking: {top5_accuracy:.4f}")
print(f"Errori Top-5 con reranking: {len(errors_reranked)}")

Numero totale domande: 300
Top-1 retrieval accuracy con reranking: 0.7833
Top-3 retrieval accuracy con reranking: 0.9500
Top-5 retrieval accuracy con reranking: 0.9633
Errori Top-5 con reranking: 11


We can now also compute the **answer containment accuracy with reranking**, namely check whether the gold answer is contained in the reranked chunks:

In [21]:
answer_top1_correct = 0
answer_top3_correct = 0
answer_top5_correct = 0
total_questions = 0

for section in data["data"]:
    for paragraph in section["paragraphs"]:
        for qa in paragraph["qas"]:
            question = qa["question"]
            gold_answers = [answer["text"] for answer in qa["answers"]]

            results = retrieve_faiss_reranked(
                question,
                faiss_top_k=20,
                rerank_top_k=5
            )

            total_questions += 1

            contexts_top1 = [result["context"] for result in results[:1]]
            contexts_top3 = [result["context"] for result in results[:3]]
            contexts_top5 = [result["context"] for result in results[:5]]

            if any(
                gold_answer in context
                for gold_answer in gold_answers
                for context in contexts_top1
            ):
                answer_top1_correct += 1

            if any(
                gold_answer in context
                for gold_answer in gold_answers
                for context in contexts_top3
            ):
                answer_top3_correct += 1

            if any(
                gold_answer in context
                for gold_answer in gold_answers
                for context in contexts_top5
            ):
                answer_top5_correct += 1

print(f"Answer containment Top-1 con reranking: {answer_top1_correct / total_questions:.4f}")
print(f"Answer containment Top-3 con reranking: {answer_top3_correct / total_questions:.4f}")
print(f"Answer containment Top-5 con reranking: {answer_top5_correct / total_questions:.4f}")

Answer containment Top-1 con reranking: 0.9333
Answer containment Top-3 con reranking: 0.9800
Answer containment Top-5 con reranking: 0.9900


## **Comparison between retrieval without reranking and retrieval with reranking**

The results show that reranking significantly improves the ordering of the retrieved chunks. The `Top-1 retrieval accuracy` increases from `0.6600` to `0.7833`, while the `Top-5 retrieval accuracy` increases from `0.9167` to `0.9633`. The number of Top-5 errors is also reduced, from `25` to `11`.

An even more evident improvement can be observed in the `Answer containment accuracy`: with reranking, the probability that the first chunk contains the correct answer increases from `0.8167` to `0.9333`. This indicates that the reranker is able to move more useful chunks for the BERT reader to the top positions, increasing the probability that the model receives a context that actually contains the answer.

| Metric | FAISS without reranking | FAISS with reranking | Improvement |
|---|---:|---:|---:|
| Top-1 retrieval accuracy | 0.6600 | 0.7833 | +0.1233 |
| Top-3 retrieval accuracy | 0.8767 | 0.9500 | +0.0733 |
| Top-5 retrieval accuracy | 0.9167 | 0.9633 | +0.0466 |
| Top-5 errors | 25 | 11 | -14 |
| Answer containment Top-1 | 0.8167 | 0.9333 | +0.1166 |
| Answer containment Top-3 | 0.9300 | 0.9800 | +0.0500 |
| Answer containment Top-5 | 0.9567 | 0.9900 | +0.0333 |

Overall, reranking reduces retriever errors and improves the quality of the contexts passed to the read comprehension model.

# **Read comprehension with BERT**

After generating the QA dataset in SQuAD-like format, a review phase was performed on the extractive answers.

This is motivated by the fact that, in some cases, the same question may allow multiple correct answers, all literally present in the text. For example, an answer such as `nell'isola di Ustica` can be considered equivalent to `isola di Ustica` or simply `Ustica`, while `dal lato sud-ovest` can also be expressed as `sud-ovest`.

For this reason, the original dataset was enriched by adding plausible alternative answers where appropriate.

This review is especially useful during the reader model evaluation phase. In fact, metrics such as Exact Match and token-level F1 can penalize predictions that are semantically correct but slightly different in the selected span. Adding alternative answers therefore enables a fairer evaluation, without making the dataset less rigorous.

The review does not modify the FAISS retrieval task, since in that case the objective is to retrieve the correct chunk associated with the question. It becomes more relevant when evaluating an extractive model, such as BERT fine-tuned on SQuAD, which must identify the exact answer span within the context.

In [ ]:
# @title
input_path = Path("/content/drive/MyDrive/nlp_project/data/qa_dataset_squad_like.json")
output_path = Path("/content/drive/MyDrive/nlp_project/data/qa_dataset_squad_like_with_alternative_answers.json")
report_path = Path("/content/drive/MyDrive/nlp_project/data/qa_alt_answers_report.json")


SYSTEM_PROMPT = """
Sei un assistente esperto nella revisione di dataset di Question Answering estrattivo in formato SQuAD.

Riceverai in input un oggetto JSON contenente:
- context
- qas
- question
- answers
- answer_start

Il tuo compito è aggiungere, solo quando necessario, risposte alternative plausibili per ogni domanda.

Regole obbligatorie:

1. Non modificare il testo del context.
2. Non modificare la domanda.
3. Non eliminare la risposta già presente.
4. Non modificare is_impossible.
5. Puoi aggiungere nuove risposte dentro il campo answers solo se sono alternative equivalenti alla risposta originale.
6. Ogni risposta alternativa deve essere una sottostringa esatta presente nel context.
7. Ogni answer_start deve indicare l'indice esatto di inizio della risposta nel context.
8. Non aggiungere parafrasi che non compaiono letteralmente nel context.
9. Non aggiungere risposte troppo generiche o ambigue.
10. Non aggiungere una risposta alternativa se peggiora la precisione della risposta.
11. Se non ci sono alternative utili, lascia answers invariato.
12. Mantieni lo stesso formato JSON dell'input.
13. Restituisci solo JSON valido, senza spiegazioni, senza markdown e senza testo aggiuntivo.

Aggiungi alternative solo nei seguenti casi:

A. Articoli o preposizioni rimovibili:
- "nell'isola di Ustica" può avere anche "isola di Ustica" o "Ustica"
- "dal lato sud-ovest" può avere anche "sud-ovest"
- "nel 1341" può avere anche "1341"
- "il 4 maggio del 1396" può avere anche "4 maggio del 1396"

B. Titoli davanti ai nomi:
- "il Califfo Al-Moez" può avere anche "Califfo Al-Moez" o "Al-Moez"
- "il Conte Ruggiero" può avere anche "Conte Ruggiero"
- "Papa Alessandro II" può avere anche "Alessandro II"

C. Date:
- se la risposta contiene una data con articolo o preposizione, puoi aggiungere la forma senza articolo/preposizione.
- esempio: "nel settembre del 1398" può avere anche "settembre del 1398"
- esempio: "l'anno 1756" può avere anche "1756"

D. Luoghi:
- se la risposta contiene una preposizione iniziale, puoi aggiungere la forma senza preposizione.
- esempio: "nella Rocca d'Aci" può avere anche "Rocca d'Aci"
- esempio: "sulla pianura di Mascali" può avere anche "pianura di Mascali"

E. Nomi propri:
- puoi aggiungere una forma più breve solo se non è ambigua nel context.
- esempio: "il Califfo Al-Moez" può avere anche "Al-Moez"
- non aggiungere alternative troppo generiche come "Giovanni", "Federico", "Martino" se nel context compaiono più persone o se il nome può essere ambiguo.

Non fare queste cose:
- Non trasformare "un'immensa eruzione sottomarina" in una parafrasi.
- Non aggiungere risposte che non sono presenti letteralmente nel context.
- Non aggiungere risposte troppo corte se perdono informazione.
- Non aggiungere solo il numero di un anno quando la domanda chiede una data completa.
- Non aggiungere solo il cognome se nel context ci sono persone diverse con cognomi simili.
- Non aggiungere alternative duplicate.
"""

def extract_json(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "", 1).strip()
    if text.startswith("```"):
        text = text.replace("```", "", 1).strip()
    if text.endswith("```"):
        text = text[:-3].strip()

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("Nessun JSON trovato nella risposta del modello")

    return text[start:end + 1]

def call_azure_on_paragraph(paragraph, max_retries=3):
    payload = {
        "context": paragraph["context"],
        "qas": paragraph["qas"]
    }

    user_prompt = "Elabora il seguente JSON:\n\n" + json.dumps(payload, ensure_ascii=False, indent=2)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=deployment_name,
                messages=[
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": user_prompt
                    }
                ],
                temperature=0,
                response_format={
                    "type": "json_object"
                }
            )

            content = response.choices[0].message.content
            parsed = json.loads(extract_json(content))

            if "context" not in parsed or "qas" not in parsed:
                raise ValueError("JSON restituito senza context o qas")

            return parsed

        except Exception as e:
            print(f"Errore tentativo {attempt + 1}/{max_retries}: {e}")
            time.sleep(2 * (attempt + 1))

    return payload

def validate_answers(paragraph):
    context = paragraph["context"]
    report_items = []

    for qa in paragraph.get("qas", []):
        old_answers = qa.get("answers", [])
        valid_answers = []
        seen = set()

        for ans in old_answers:
            if not isinstance(ans, dict):
                continue

            text = ans.get("text", "")
            start = ans.get("answer_start", None)

            if not isinstance(text, str):
                continue

            text = text.strip()

            if not text:
                continue

            if not isinstance(start, int):
                start = context.find(text)

            if start < 0 or context[start:start + len(text)] != text:
                start = context.find(text)

            if start < 0:
                continue

            if context[start:start + len(text)] != text:
                continue

            key = (text, start)

            if key in seen:
                continue

            seen.add(key)

            valid_answers.append({
                "text": text,
                "answer_start": start
            })

        original_count = len(qa.get("answers", []))
        qa["answers"] = valid_answers

        report_items.append({
            "id": qa.get("id"),
            "question": qa.get("question"),
            "answers_after_validation": valid_answers,
            "num_answers_after_validation": len(valid_answers),
            "num_answers_before_validation": original_count
        })

    return paragraph, report_items

def process_dataset(dataset):
    new_dataset = {
        "version": dataset.get("version", "1.0"),
        "data": []
    }

    global_report = []
    total_paragraphs = 0
    total_qas = 0
    total_answers_before = 0
    total_answers_after = 0

    for data_idx, item in enumerate(dataset.get("data", [])):
        new_item = {
            "title": item.get("title", ""),
            "paragraphs": []
        }

        for par_idx, paragraph in enumerate(item.get("paragraphs", [])):
            total_paragraphs += 1

            print(f"Elaboro sezione {data_idx + 1}/{len(dataset.get('data', []))}, paragrafo {par_idx + 1}/{len(item.get('paragraphs', []))}")

            old_paragraph = json.loads(json.dumps(paragraph, ensure_ascii=False))
            enriched = call_azure_on_paragraph(paragraph)
            validated, report_items = validate_answers(enriched)

            old_qas_by_id = {
                qa.get("id"): qa
                for qa in old_paragraph.get("qas", [])
            }

            for qa in validated.get("qas", []):
                qa_id = qa.get("id")
                old_qa = old_qas_by_id.get(qa_id, {})
                old_answers = old_qa.get("answers", [])
                new_answers = qa.get("answers", [])

                total_qas += 1
                total_answers_before += len(old_answers)
                total_answers_after += len(new_answers)

                old_answer_keys = {
                    (ans.get("text"), ans.get("answer_start"))
                    for ans in old_answers
                }

                added_answers = [
                    ans for ans in new_answers
                    if (ans.get("text"), ans.get("answer_start")) not in old_answer_keys
                ]

                if added_answers:
                    global_report.append({
                        "title": item.get("title", ""),
                        "paragraph_index": par_idx,
                        "id": qa_id,
                        "question": qa.get("question"),
                        "original_answers": old_answers,
                        "added_answers": added_answers,
                        "final_answers": new_answers
                    })

            new_item["paragraphs"].append(validated)

            time.sleep(0.5)

        new_dataset["data"].append(new_item)

    summary = {
        "total_paragraphs": total_paragraphs,
        "total_qas": total_qas,
        "total_answers_before": total_answers_before,
        "total_answers_after": total_answers_after,
        "total_added_answers": total_answers_after - total_answers_before,
        "modified_qas": len(global_report)
    }

    return new_dataset, {
        "summary": summary,
        "changes": global_report
    }

dataset = json.loads(input_path.read_text(encoding="utf-8"))

new_dataset, report = process_dataset(dataset)

output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    json.dump(new_dataset, f, ensure_ascii=False, indent=2)

with report_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("Dataset saved to:", output_path)
print("Report saved to:", report_path)
print(json.dumps(report["summary"], ensure_ascii=False, indent=2))

To make the pipeline evaluation more realistic, the Question Answering dataset was extended by also introducing **unanswerable** questions, namely questions for which no answer exists in the document corpus used by the system.

In the original dataset, each question was associated with an answer literally extracted from the corresponding context. However, in a real scenario, the user may also ask out-of-domain questions or questions for which the book does not contain sufficient information. In these cases, the system should not force the extraction of an answer from an irrelevant chunk, but should be able to recognize that the answer is not present.

The unanswerable questions were constructed so that the answer was not absent only from the associated single chunk, but from the entire corpus. This choice is important because the pipeline does not work on an isolated context: it first retrieves the most relevant chunks through FAISS, then reorders them with the reranker, and finally applies the read comprehension model. Consequently, a question can be considered truly unanswerable only if no passage in the book contains the information needed to answer it.

*Moreover, unanswerable questions must not be off-topic, but semantically related to their corresponding chunk. They must look like questions that a user might ask after reading that section, but for which the text does not provide an answer.*

In the SQuAD-like format, these questions are represented by setting the `is_impossible` field to `true` and leaving the `answers` list empty. In this way, it is possible to evaluate not only the system’s ability to correctly extract an answer when it is present, but also its ability to abstain when the corpus does not contain the answer.

The dataset was extended by adding 80 unanswerable questions, corresponding to about 26.67% of the final dataset. This choice makes it possible to evaluate the system’s ability to recognize cases in which the corpus does not contain an answer, while still keeping the dataset mainly oriented toward extractive Question Answering on answerable questions.

In [ ]:
# @title
input_path = Path("/content/drive/MyDrive/nlp_project/data/qa_dataset_squad_like_with_alternative_answers.json")
output_path = Path("/content/drive/MyDrive/nlp_project/data/qa_dataset_squad_like_final.json")
report_path = Path("/content/drive/MyDrive/nlp_project/data/unanswerable_questions_report.json")

dataset = json.loads(input_path.read_text(encoding="utf-8"))

all_contexts = []

for section_idx, section in enumerate(dataset["data"]):
    for paragraph_idx, paragraph in enumerate(section["paragraphs"]):
        all_contexts.append({
            "section_idx": section_idx,
            "paragraph_idx": paragraph_idx,
            "title": section["title"],
            "context": paragraph["context"]
        })

full_corpus_text = "\n\n".join(
    item["title"] + "\n" + item["context"]
    for item in all_contexts
)

def build_unanswerable_prompt(context_item):
    return f"""
Sei un esperto nella costruzione di dataset di Question Answering in italiano in formato SQuAD 2.0.

Devi generare una sola domanda non answerable per il chunk fornito.

Una domanda non answerable è una domanda per cui la risposta non è presente nel chunk specifico e non è presente neppure nell'intero corpus documentale.

La domanda deve rispettare due condizioni contemporaneamente:

1. Deve essere semanticamente collegata al chunk fornito.
2. Deve richiedere un'informazione che il chunk non contiene e che non è presente nel corpus.

Quindi non devi generare domande fuori tema o casuali. La domanda deve sembrare plausibile per un utente che sta leggendo una sezione simile, ma il testo non deve fornire la risposta.

REGOLE OBBLIGATORIE:
1. La domanda deve essere coerente con il tema del chunk.
2. La domanda deve essere plausibile nel dominio storico, geografico, religioso, geologico, militare, architettonico o culturale del Castello di Aci Castello.
3. La risposta non deve essere presente nel chunk.
4. La risposta non deve essere deducibile direttamente dal chunk.
5. La risposta non deve essere presente nel resto del corpus.
6. Non generare domande assurde, ironiche o chiaramente false.
7. Non generare domande troppo generiche.
8. Non usare formule come "secondo il testo", "nel brano", "nel contesto", "nel documento".
9. La domanda deve essere autonoma e comprensibile senza leggere il chunk.
10. Non generare domande identiche o troppo simili alle domande già presenti nel dataset.
11. Restituisci solo JSON valido, senza markdown e senza testo aggiuntivo.

ESEMPI DI DOMANDE BUONE:
- Se il chunk parla della rupe e della sua origine geologica, puoi chiedere quale studio moderno abbia datato la rupe, se questa informazione non è presente.
- Se il chunk parla di una battaglia medievale, puoi chiedere quale cronaca militare descriva le perdite, se questa informazione non è presente.
- Se il chunk parla di un restauro o di rovine, puoi chiedere quale architetto abbia diretto un intervento specifico, se questa informazione non è presente.
- Se il chunk parla di una chiesa o di un santuario, puoi chiedere quale documento ecclesiastico ne abbia regolato una fase successiva, se questa informazione non è presente.
- Se il chunk parla di un personaggio storico, puoi chiedere un dettaglio biografico plausibile ma non riportato dal corpus.
- Se il chunk parla di una dominazione, puoi chiedere un documento, un trattato o una fonte non menzionata nel corpus.

ESEMPI DI DOMANDE DA EVITARE:
- domande su calcio, politica moderna, cucina o argomenti estranei;
- domande la cui risposta è contenuta nel chunk;
- domande la cui risposta è contenuta altrove nel corpus;
- domande troppo vaghe come "Cosa successe dopo?";
- domande troppo evidentemente false o assurde;
- domande che chiedono informazioni turistiche moderne se il chunk parla di eventi antichi, a meno che il collegamento non sia plausibile.

Formato richiesto:
{{
  "question": "..."
}}

SEZIONE:
{context_item["title"]}

CHUNK:
{context_item["context"]}
"""

def extract_json(text):
    text = text.strip()

    if text.startswith("```json"):
        text = text.replace("```json", "", 1).strip()

    if text.startswith("```"):
        text = text.replace("```", "", 1).strip()

    if text.endswith("```"):
        text = text[:-3].strip()

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("Nessun JSON trovato nella risposta del modello")

    return text[start:end + 1]

def normalize_text(text):
    return " ".join(text.lower().split())

def is_question_valid(question, existing_questions):
    q_norm = normalize_text(question)

    if not question.endswith("?"):
        return False

    if q_norm in existing_questions:
        return False

    if len(question.split()) < 6:
        return False

    forbidden_phrases = [
        "secondo il testo",
        "nel testo",
        "nel brano",
        "nel contesto",
        "nel documento",
        "secondo il brano",
        "secondo il contesto",
        "secondo il documento"
    ]

    if any(phrase in q_norm for phrase in forbidden_phrases):
        return False

    return True

def generate_unanswerable_question_for_context(context_item, max_retries=3):
    prompt = build_unanswerable_prompt(context_item)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=deployment_name,
                messages=[
                    {
                        "role": "system",
                        "content": "Genera esclusivamente JSON valido con una domanda non answerable per il chunk fornito."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0.4,
                response_format={"type": "json_object"}
            )

            content = response.choices[0].message.content
            parsed = json.loads(extract_json(content))

            question = parsed.get("question", "")

            if isinstance(question, str):
                return question.strip()

            return ""

        except Exception as e:
            print(f"Errore tentativo {attempt + 1}/{max_retries}: {e}")
            time.sleep(2 * (attempt + 1))

    return ""

def generate_unanswerable_questions(target_num=80):
    existing_questions = set()

    for section in dataset["data"]:
        for paragraph in section["paragraphs"]:
            for qa in paragraph["qas"]:
                existing_questions.add(normalize_text(qa["question"]))

    selected_contexts = random.sample(
        all_contexts,
        k=min(target_num, len(all_contexts))
    )

    generated_items = []

    for context_item in selected_contexts:
        if len(generated_items) >= target_num:
            break

        question = generate_unanswerable_question_for_context(context_item)

        if is_question_valid(question, existing_questions):
            q_norm = normalize_text(question)
            existing_questions.add(q_norm)

            generated_items.append({
                "question": question,
                "section_idx": context_item["section_idx"],
                "paragraph_idx": context_item["paragraph_idx"],
                "title": context_item["title"]
            })

        print(f"Domande unanswerable generate: {len(generated_items)}/{target_num}")

        time.sleep(0.5)

    return generated_items

unanswerable_questions = generate_unanswerable_questions(
    target_num=80
)

for i, item in enumerate(unanswerable_questions, start=1):
    qa = {
        "id": f"unanswerable_{i:04d}",
        "question": item["question"],
        "answers": [],
        "is_impossible": True
    }

    dataset["data"][item["section_idx"]]["paragraphs"][item["paragraph_idx"]]["qas"].append(qa)

output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

report = {
    "num_unanswerable_added": len(unanswerable_questions),
    "questions": [
        {
            "id": f"unanswerable_{i:04d}",
            "section_title": item["title"],
            "section_idx": item["section_idx"],
            "paragraph_idx": item["paragraph_idx"],
            "question": item["question"]
        }
        for i, item in enumerate(unanswerable_questions, start=1)
    ]
}

with report_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

total_qas = 0
answerable_qas = 0
unanswerable_qas = 0

for section in dataset["data"]:
    for paragraph in section["paragraphs"]:
        for qa in paragraph["qas"]:
            total_qas += 1

            if qa.get("is_impossible", False):
                unanswerable_qas += 1
            else:
                answerable_qas += 1

print("Dataset saved to:", output_path)
print("Report saved to:", report_path)
print("Domande answerable:", answerable_qas)
print("Domande unanswerable:", unanswerable_qas)
print("Totale domande:", total_qas)

Let us see an example of an unanswerable question:

In [ ]:
# @title
qa_path = Path("/content/drive/MyDrive/nlp_project/data/qa_dataset_squad_like_final.json")

data = json.loads(qa_path.read_text(encoding="utf-8"))

input_chunk_id = "chunk_0015"

unanswerable_examples = []
chunk_found = False

for section in data["data"]:
    section_title = section["title"]

    for paragraph in section["paragraphs"]:
        context = paragraph["context"]

        for qa in paragraph["qas"]:
            question_id = qa["id"]

            if "_q" in question_id:
                current_chunk_id = question_id.split("_q")[0]
            elif "_unanswerable" in question_id:
                current_chunk_id = question_id.split("_unanswerable")[0]
            else:
                current_chunk_id = question_id

            if current_chunk_id != input_chunk_id:
                continue

            chunk_found = True

            if qa.get("is_impossible", False):
                unanswerable_examples.append({
                    "id": qa["id"],
                    "question": qa["question"],
                    "section_title": section_title,
                    "chunk_id": current_chunk_id,
                    "context": context
                })

if not chunk_found:
    safe_input_chunk_id = html.escape(str(input_chunk_id))

    html_card = f"""
    <div style="font-family:Arial, sans-serif; border:1px solid #ddd; border-radius:16px; padding:22px; background:#fafafa; margin-top:15px;">
        <div style="font-size:22px; font-weight:700; margin-bottom:12px;">
            Chunk not found
        </div>
        <div style="color:#666; font-size:15px;">
            No question associated with chunk <b>{safe_input_chunk_id}</b> was found in the dataset.
        </div>
    </div>
    """

    print(f"Chunk trovato: {chunk_found}")
    print("Numero di domande non answerable trovate per il chunk fornito: 0")
    display(HTML(html_card))

elif len(unanswerable_examples) == 0:
    safe_input_chunk_id = html.escape(str(input_chunk_id))

    html_card = f"""
    <div style="font-family:Arial, sans-serif; border:1px solid #ddd; border-radius:16px; padding:22px; background:#fafafa; margin-top:15px;">
        <div style="font-size:22px; font-weight:700; margin-bottom:12px;">
            Unanswerable question
        </div>
        <div style="color:#666; font-size:15px;">
            The chunk <b>{safe_input_chunk_id}</b> was found, but it does not contain questions with <b>is_impossible = True</b>.
        </div>
    </div>
    """

    print(f"Chunk trovato: {chunk_found}")
    print("Numero di domande non answerable trovate per il chunk fornito: 0")
    display(HTML(html_card))

else:
    example = unanswerable_examples[0]

    safe_id = html.escape(str(example["id"]))
    safe_question = html.escape(str(example["question"]))
    safe_section = html.escape(str(example["section_title"]))
    safe_chunk_id = html.escape(str(example["chunk_id"]))

    context_text = str(example["context"])
    max_chars = 1400

    if len(context_text) > max_chars:
        context_text = context_text[:max_chars] + "..."

    safe_context = html.escape(context_text)

    html_card = f"""
    <div style="font-family:Arial, sans-serif; border:1px solid #ddd; border-radius:16px; padding:22px; background:#fafafa; margin-top:15px;">
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:18px;">
            <div style="font-size:22px; font-weight:700;">
                Unanswerable question
            </div>
            <div style="background:#ffe8e8; color:#a40000; border:1px solid #ffb5b5; border-radius:999px; padding:7px 13px; font-size:13px; font-weight:700;">
                NO ANSWER
            </div>
        </div>

        <div style="display:flex; gap:12px; flex-wrap:wrap; margin-bottom:20px;">
            <div style="background:white; border:1px solid #ddd; border-radius:10px; padding:10px 14px;">
                <div style="font-size:12px; color:#777;">Question ID</div>
                <div style="font-weight:700;">{safe_id}</div>
            </div>

            <div style="background:white; border:1px solid #ddd; border-radius:10px; padding:10px 14px;">
                <div style="font-size:12px; color:#777;">Chunk ID</div>
                <div style="font-weight:700;">{safe_chunk_id}</div>
            </div>

            <div style="background:white; border:1px solid #ddd; border-radius:10px; padding:10px 14px;">
                <div style="font-size:12px; color:#777;">Section</div>
                <div style="font-weight:700;">{safe_section}</div>
            </div>

            <div style="background:white; border:1px solid #ddd; border-radius:10px; padding:10px 14px;">
                <div style="font-size:12px; color:#777;">is_impossible</div>
                <div style="font-weight:700;">True</div>
            </div>

            <div style="background:white; border:1px solid #ddd; border-radius:10px; padding:10px 14px;">
                <div style="font-size:12px; color:#777;">answers</div>
                <div style="font-weight:700;">[]</div>
            </div>
        </div>

        <div style="background:white; border:1px solid #ddd; border-radius:14px; padding:18px; margin-bottom:18px;">
            <div style="font-size:13px; color:#666; margin-bottom:7px;">Question</div>
            <div style="font-size:21px; font-weight:700; line-height:1.4;">
                {safe_question}
            </div>
        </div>

        <div style="background:white; border:1px solid #ddd; border-radius:14px; padding:18px;">
            <div style="font-size:13px; color:#666; margin-bottom:9px;">Associated context</div>
            <div style="line-height:1.65; font-size:15px; white-space:pre-wrap;">
                {safe_context}
            </div>
        </div>
    </div>
    """

    print(f"Chunk trovato: {chunk_found}")
    print(f"Numero di domande non answerable trovate per il chunk fornito: {len(unanswerable_examples)}")
    display(HTML(html_card))

We now initialize the extractive Question Answering model used in the read comprehension phase. The model used is `deepset/xlm-roberta-large-squad2`, a multilingual model for extractive Question Answering based on XLM-RoBERTa Large. The architecture consists of a Transformer encoder with **24** layers, hidden size **1024**, and **16** attention heads. The model uses a SentencePiece tokenizer with a multilingual vocabulary and derives from XLM-RoBERTa, pre-trained on 2.5 TB of CommonCrawl data in 100 languages. Since it is fine-tuned on SQuAD 2.0, the model is also able to handle unanswerable questions.

The model is loaded through the Hugging Face `pipeline` function with the `"question-answering"` task. In this way, given a question and a textual context as input, the model returns the text span that it considers the most likely answer.

The cell automatically selects the execution device: if a CUDA GPU is available, the model is run on the GPU; otherwise, the CPU is used. Finally, a confirmation message and the actual device used are printed.

In [22]:
qa_reader = pipeline(
    "question-answering",
    model="deepset/xlm-roberta-large-squad2",
    tokenizer="deepset/xlm-roberta-large-squad2",
    device=0 if torch.cuda.is_available() else -1,
    handle_impossible_answer=True
)

print("QA model loaded")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

config.json:   0%|          | 0.00/606 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaForQuestionAnswering LOAD REPORT from: deepset/xlm-roberta-large-squad2
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/179 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

QA model loaded
Device: cuda


The following method receives a question as input and uses a pipeline composed of retrieval, reranking, and extractive read comprehension with handling of no-answer cases.

First, the `retrieve_faiss_reranked` function retrieves an initial set of candidate chunks through FAISS. In this phase, the first `faiss_top_k` results are considered, in order to obtain a sufficiently broad set of potentially relevant contexts. Then, these chunks are reordered by the reranker, which evaluates the relevance of each question-context pair more accurately. After reranking, only the first `reader_top_k` chunks are kept, namely those considered most suitable for the automatic reading phase.

For each reranked chunk, the extractive Question Answering model is applied to the question-context pair. In this version, the reader is run with `handle_impossible_answer=True`, so it is not forced to always extract a span from the text. The model can return a textual answer when it identifies a plausible span, or an empty answer when it believes that the context does not contain the requested information.

The candidate answers are saved together with useful information for analysis, including the reader score, the extracted span position, the rank after reranking, the chunk identifier, the corresponding section, the original FAISS retriever score, the score assigned by the reranker, the textual context used, the boolean field `no_answer`, which indicates whether the reader produced an empty answer, and some combined scores useful for the final selection.

In particular, the following value is computed:

$$
s_{\text{BERT-rank}} = s_{\text{reader}} \times s_{\text{reranker}}
$$

This score combines the reader confidence on the extracted answer with the relevance of the chunk according to the reranker. In this way, the final answer is not selected only according to the confidence of the BERT model, but also by considering the quality of the context from which it was extracted.

If the reader produces one or more non-empty answers, the pipeline selects the answer with the maximum value of `reader_score × reranker_score`. The answer is accepted only if this value exceeds the `min_bert_rank_score` threshold; otherwise, the system returns a `no_answer` result, indicating that the information is not sufficiently supported by the retrieved chunks.

If all the best answers produced by the reader are empty, the function does not immediately return `no_answer`. First, it identifies the chunk with the highest value of:

$$
s_{\text{combined}} = s_{\text{FAISS}} \times s_{\text{reranker}}
$$

This score measures how relevant a chunk is considered by both the vector retriever and the reranker. If the selected chunk has a combined score higher than the `min_forced_product_score` threshold, and if the reader is not too confident in the empty answer, namely if the score of the empty answer is less than or equal to `max_empty_answer_score`, then the pipeline forces the selection of that alternative answer.

This mechanism is used to reduce false `no_answer` cases, namely situations in which the answer is present in the corpus but the reader assigns the empty answer as its first choice. At the same time, the use of thresholds avoids forcing answers when the chunk is not relevant enough or when the reader is highly confident that the answer is not present.

In summary, the function adopts a decision strategy based on three elements: reader confidence, chunk relevance according to the reranker, and semantic similarity provided by FAISS. The goal is to balance the ability to extract correct answers with the ability to abstain when the corpus does not contain sufficient information.

In [23]:
def answer_question_with_bert(
    question,
    faiss_top_k=20,
    reader_top_k=5,
    min_bert_rank_score=0.25,
    min_forced_product_score=0.68,
    max_empty_answer_score=0.55
):
    retrieved_chunks = retrieve_faiss_reranked(
        question,
        faiss_top_k=faiss_top_k,
        rerank_top_k=reader_top_k
    )

    questions = [question for _ in retrieved_chunks]
    contexts = [chunk["context"] for chunk in retrieved_chunks]

    raw_results = qa_reader(
        question=questions,
        context=contexts,
        handle_impossible_answer=True,
        batch_size=reader_top_k,
        top_k=2
    )

    candidate_answers = []

    for rank, (chunk, result_group) in enumerate(zip(retrieved_chunks, raw_results), start=1):
        if isinstance(result_group, dict):
            result_group = [result_group]

        best_result = result_group[0]
        answer = best_result["answer"].strip()

        reader_score = float(best_result["score"])
        retrieval_score = float(chunk["score"])
        reranker_score = float(chunk["reranker_score"])
        bert_rank_score = reader_score * reranker_score

        second_valid_result = None

        for result in result_group[1:]:
            if result["answer"].strip() != "":
                second_valid_result = result
                break

        candidate_answers.append({
            "answer": answer,
            "score": reader_score,
            "start": int(best_result["start"]),
            "end": int(best_result["end"]),
            "reranked_rank": rank,
            "chunk_id": chunk["chunk_id"],
            "section_title": chunk["section_title"],
            "retrieval_score": retrieval_score,
            "reranker_score": reranker_score,
            "combined_score": retrieval_score * reranker_score,
            "bert_rank_score": bert_rank_score,
            "context": chunk["context"],
            "no_answer": answer == "",
            "second_valid_answer": second_valid_result
        })

    valid_answers = [
        candidate
        for candidate in candidate_answers
        if candidate["answer"] != ""
    ]

    if len(valid_answers) == 0:
        best_forced_candidate = max(
            candidate_answers,
            key=lambda x: x["combined_score"]
        )

        second_valid_result = best_forced_candidate["second_valid_answer"]

        can_force_answer = (
            second_valid_result is not None
            and best_forced_candidate["combined_score"] >= min_forced_product_score
            and best_forced_candidate["score"] <= max_empty_answer_score
        )

        if can_force_answer:
            forced_reader_score = float(second_valid_result["score"])
            forced_bert_rank_score = forced_reader_score * best_forced_candidate["reranker_score"]

            return {
                "answer": second_valid_result["answer"].strip(),
                "score": forced_reader_score,
                "start": int(second_valid_result["start"]),
                "end": int(second_valid_result["end"]),
                "reranked_rank": best_forced_candidate["reranked_rank"],
                "chunk_id": best_forced_candidate["chunk_id"],
                "section_title": best_forced_candidate["section_title"],
                "retrieval_score": best_forced_candidate["retrieval_score"],
                "reranker_score": best_forced_candidate["reranker_score"],
                "combined_score": best_forced_candidate["combined_score"],
                "bert_rank_score": forced_bert_rank_score,
                "empty_answer_score": best_forced_candidate["score"],
                "context": best_forced_candidate["context"],
                "no_answer": False,
                "forced_by_retrieval_reranker": True,
                "selected_by_bert_rank_score": False,
                "rejected_by_bert_rank_score": False
            }, candidate_answers

        return {
            "answer": "The answer is not present in the context.",
            "score": best_forced_candidate["score"],
            "start": None,
            "end": None,
            "reranked_rank": best_forced_candidate["reranked_rank"],
            "chunk_id": best_forced_candidate["chunk_id"],
            "section_title": best_forced_candidate["section_title"],
            "retrieval_score": best_forced_candidate["retrieval_score"],
            "reranker_score": best_forced_candidate["reranker_score"],
            "combined_score": best_forced_candidate["combined_score"],
            "bert_rank_score": best_forced_candidate["bert_rank_score"],
            "empty_answer_score": best_forced_candidate["score"],
            "context": best_forced_candidate["context"],
            "no_answer": True,
            "forced_by_retrieval_reranker": False,
            "selected_by_bert_rank_score": False,
            "rejected_by_bert_rank_score": False
        }, candidate_answers


    best_answer = max(valid_answers, key=lambda x: x["bert_rank_score"])
    selected_by_bert_rank_score = len(valid_answers) >= 2

    if best_answer["bert_rank_score"] < min_bert_rank_score:
        return {
            "answer": "The answer is not present in the context.",
            "score": best_answer["score"],
            "start": None,
            "end": None,
            "reranked_rank": best_answer["reranked_rank"],
            "chunk_id": best_answer["chunk_id"],
            "section_title": best_answer["section_title"],
            "retrieval_score": best_answer["retrieval_score"],
            "reranker_score": best_answer["reranker_score"],
            "combined_score": best_answer["combined_score"],
            "bert_rank_score": best_answer["bert_rank_score"],
            "context": best_answer["context"],
            "no_answer": True,
            "forced_by_retrieval_reranker": False,
            "selected_by_bert_rank_score": False,
            "rejected_by_bert_rank_score": True
        }, candidate_answers

    best_answer["no_answer"] = False
    best_answer["forced_by_retrieval_reranker"] = False
    best_answer["selected_by_bert_rank_score"] = selected_by_bert_rank_score
    best_answer["rejected_by_bert_rank_score"] = False

    return best_answer, candidate_answers

We run a test example:

In [ ]:
# @title
question = "Chi ordinò la distruzione del castello nel 902 d.C?"

best_answer, candidate_answers = answer_question_with_bert(
    question,
    faiss_top_k=20,
    reader_top_k=5
)

answer_status = "NO ANSWER" if best_answer["no_answer"] else "ANSWER"

safe_question = html.escape(question)
safe_answer = html.escape(str(best_answer["answer"]))
safe_chunk = html.escape(str(best_answer["chunk_id"]))
safe_section = html.escape(str(best_answer["section_title"]))
safe_context = html.escape(str(best_answer["context"]))

final_answer_html = f"""
<div style="border:1px solid #ddd; border-radius:14px; padding:20px; margin-bottom:22px; background:#fafafa; font-family:Arial, sans-serif;">
    <div style="font-size:13px; color:#666; margin-bottom:6px;">Question</div>
    <div style="font-size:19px; font-weight:600; margin-bottom:18px;">{safe_question}</div>

    <div style="font-size:13px; color:#666; margin-bottom:6px;">Final answer</div>
    <div style="font-size:25px; font-weight:700; margin-bottom:18px;">{safe_answer}</div>

    <div style="display:flex; gap:12px; flex-wrap:wrap;">
        <div style="background:white; border:1px solid #ddd; border-radius:9px; padding:10px 14px;">
            <div style="font-size:12px; color:#777;">Stato</div>
            <div style="font-weight:700;">{answer_status}</div>
        </div>
        <div style="background:white; border:1px solid #ddd; border-radius:9px; padding:10px 14px;">
            <div style="font-size:12px; color:#777;">Chunk</div>
            <div style="font-weight:700;">{safe_chunk}</div>
        </div>
        <div style="background:white; border:1px solid #ddd; border-radius:9px; padding:10px 14px;">
            <div style="font-size:12px; color:#777;">Section</div>
            <div style="font-weight:700;">{safe_section}</div>
        </div>
        <div style="background:white; border:1px solid #ddd; border-radius:9px; padding:10px 14px;">
            <div style="font-size:12px; color:#777;">Reader score</div>
            <div style="font-weight:700;">{best_answer["score"]:.4f}</div>
        </div>
        <div style="background:white; border:1px solid #ddd; border-radius:9px; padding:10px 14px;">
            <div style="font-size:12px; color:#777;">FAISS score</div>
            <div style="font-weight:700;">{best_answer["retrieval_score"]:.4f}</div>
        </div>
        <div style="background:white; border:1px solid #ddd; border-radius:9px; padding:10px 14px;">
            <div style="font-size:12px; color:#777;">Reranker score</div>
            <div style="font-weight:700;">{best_answer["reranker_score"]:.4f}</div>
        </div>
    </div>
</div>
"""

display(HTML(final_answer_html))

candidates_df = pd.DataFrame(candidate_answers)

candidates_df["Candidate answer"] = candidates_df["answer"].apply(
    lambda x: "[EMPTY]" if str(x).strip() == "" else str(x)
)

candidates_df["Reader score"] = candidates_df["score"].astype(float)
candidates_df["FAISS score"] = candidates_df["retrieval_score"].astype(float)
candidates_df["Reranker score"] = candidates_df["reranker_score"].astype(float)

candidates_view = candidates_df[
    [
        "reranked_rank",
        "chunk_id",
        "section_title",
        "Candidate answer",
        "Reader score",
        "no_answer",
        "FAISS score",
        "Reranker score"
    ]
].rename(columns={
    "reranked_rank": "Rank",
    "chunk_id": "Chunk",
    "section_title": "Section",
    "no_answer": "No answer"
})

styled_candidates = (
    candidates_view
    .style
    .format({
        "Reader score": "{:.4f}",
        "FAISS score": "{:.4f}",
        "Reranker score": "{:.4f}"
    })
    .background_gradient(subset=["Reader score"])
    .background_gradient(subset=["FAISS score"])
    .background_gradient(subset=["Reranker score"])
    .set_properties(**{
        "text-align": "left",
        "white-space": "pre-wrap",
        "font-family": "Arial, sans-serif",
        "font-size": "13px"
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("text-align", "left"),
                ("font-weight", "bold"),
                ("background-color", "#f2f2f2"),
                ("padding", "8px"),
                ("border-bottom", "1px solid #ddd")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("padding", "8px"),
                ("border-bottom", "1px solid #eee")
            ]
        },
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("width", "100%"),
                ("margin-bottom", "22px")
            ]
        }
    ])
)

display(HTML("<h3 style='font-family:Arial, sans-serif;'>Candidate answers on retrieved chunks</h3>"))
display(styled_candidates)

best_context_html = f"""
<div style="border:1px solid #ddd; border-radius:14px; padding:20px; margin-top:22px; background:white; font-family:Arial, sans-serif;">
    <div style="font-size:20px; font-weight:700; margin-bottom:8px;">
        Context of the selected chunk
    </div>
    <div style="font-size:14px; color:#666; margin-bottom:14px;">
        {safe_chunk} · {safe_section}
    </div>
    <div style="line-height:1.65; font-size:15px; white-space:pre-wrap;">
        {safe_context}
    </div>
</div>
"""

display(HTML(best_context_html))

We now define the functions used to evaluate the answers predicted by the read comprehension model against the correct answers contained in the dataset.

The `normalize_answer` function normalizes a textual answer by converting everything to lowercase, removing punctuation, and standardizing spaces. This normalization makes the comparison between predicted and gold answers less sensitive to superficial differences, such as uppercase letters, commas, quotation marks, or multiple spaces.

The `exact_match_score` function computes the Exact Match between a prediction and a single gold answer. After normalization, it returns `1` if the two answers match exactly, otherwise it returns `0`.

The `f1_score` function computes the degree of overlap between the words of the predicted answer and those of a single gold answer. Precision and recall are computed at token level and then combined using the harmonic mean. This metric is less strict than Exact Match, because it assigns a partial score even when the predicted answer does not perfectly match the gold answer, but still shares some correct words.

The `metric_max_over_ground_truths` function extends metric computation to the case in which a question has multiple alternative gold answers. The prediction is compared separately with each correct answer, and the maximum score obtained is retained. In this way, the model is not penalized when it produces a correct answer variant, for example a form with or without an article, preposition, or title.

Therefore, for each question, the final Exact Match and final F1 are computed as the maximum score obtained against all gold answers associated with that question.

In [24]:
def normalize_answer(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    text = " ".join(text.split())
    return text

def exact_match_score(prediction, ground_truth):
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))

def f1_score(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(ground_truth).split()

    if len(pred_tokens) == 0 and len(gold_tokens) == 0:
        return 1.0

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

def metric_max_over_ground_truths(metric_fn, prediction, ground_truths):
    scores = []

    for ground_truth in ground_truths:
        score = metric_fn(prediction, ground_truth)
        scores.append(score)

    return max(scores) if scores else 0.0

We now evaluate the entire extractive Question Answering pipeline on the SQuAD-like dataset extended with alternative answers and unanswerable questions.

First, the `qa_dataset_squad_like_final.json` file is loaded. It contains both answerable questions, namely questions whose answer is present in the corpus, and unanswerable questions, namely questions for which the document corpus does not contain enough information to produce a correct answer.

For answerable questions, the dataset stores a list of possible correct answers. These alternative answers represent equivalent variants of the same span, for example forms with or without an article, preposition, or title. For unanswerable questions, instead, the `answers` field is empty and the `is_impossible` field is set to `true`, following a logic similar to SQuAD 2.0.

Then, for each question in the dataset, the `answer_question_with_bert` function is called. This function uses a pipeline composed of three phases: retrieval through FAISS, reranking of the retrieved chunks, and extractive read comprehension through BERT.

In the first phase, FAISS retrieves the first `faiss_top_k` candidate chunks, namely the passages most similar to the question according to embedding similarity. In the second phase, the reranker evaluates each question-context pair and reorders the chunks according to their relevance. After reranking, the first `reader_top_k` chunks are retained and passed to the read comprehension model.

The BERT model is then applied separately to each of the reranked chunks. Unlike the previous version of the pipeline, the reader is no longer forced to always extract an answer: thanks to the handling of impossible cases, it can also return a `no_answer` prediction when it considers that the context does not contain an answer to the question.

For each question-context pair, the reader therefore produces either a candidate answer or an empty answer, together with a confidence score and the extracted span position, when available. The final function selects the best non-empty answer when it is available and sufficiently reliable. If, instead, no chunk produces a valid answer, the system returns `unanswerable`, indicating that the answer is not present in the retrieved contexts.

The evaluation therefore distinguishes two cases. For answerable questions, the predicted answer is compared with all alternative gold answers associated with the question, using the standard `Exact Match` and `F1 Score` metrics. The computation follows a logic analogous to the official SQuAD evaluation: the prediction is compared separately with each available gold answer and, for each metric, the maximum score obtained is retained.

For unanswerable questions, instead, the prediction is considered correct only if the system actually returns a `no_answer` result. If the system extracts a span from one of the retrieved chunks, the prediction is considered wrong, because it indicates that the model produced an answer not supported by the corpus.

In addition to global `Exact Match` and `F1 Score`, specific metrics are also computed to evaluate the system’s abstention capability. In particular, the `No-answer accuracy` measures how many unanswerable questions are correctly recognized, the `False no-answer rate` measures how many answerable questions are incorrectly classified as having no answer, and the `False answer rate` measures how many unanswerable questions incorrectly receive an extracted answer.

During evaluation, several pieces of information useful for later analysis are saved for each question, including the question, the value of `is_impossible`, the list of gold answers, the predicted answer, the `predicted_no_answer` indication, the `Exact Match` and `F1` values, the reader score, the chunk rank after reranking, the chunk identifier, the section title, the original FAISS retriever score, the score assigned by the reranker, and the possible indication that the answer was forced based on the reranker score.

At the end of the cell, the mean `Exact Match` and mean `F1 Score` are computed and printed on the whole dataset, together with the specific metrics for answerable and unanswerable questions. Finally, the results are converted into a pandas DataFrame and the first rows are displayed, so that some system predictions can be qualitatively inspected.

In [25]:
qa_path = Path("/content/drive/MyDrive/nlp_project/data/qa_dataset_squad_like_final.json")

data = json.loads(qa_path.read_text(encoding="utf-8"))

evaluation_results = []

total_questions = 0
exact_match_total = 0
f1_total = 0

answerable_total = 0
unanswerable_total = 0

answerable_correct = 0
unanswerable_correct = 0

false_no_answer = 0
false_answer = 0

for section in tqdm(data["data"]):
    for paragraph in section["paragraphs"]:
        for qa in paragraph["qas"]:
            question = qa["question"]
            qid = qa["id"]
            is_impossible = qa.get("is_impossible", False)
            gold_answers = [answer["text"] for answer in qa.get("answers", [])]

            best_answer, candidate_answers = answer_question_with_bert(
                question=question,
                faiss_top_k=20,
                reader_top_k=5,
                min_bert_rank_score=0.25,
                min_forced_product_score=0.68,
                max_empty_answer_score=0.55
            )

            predicted_answer = best_answer["answer"]
            predicted_no_answer = best_answer.get("no_answer", False)

            total_questions += 1

            if is_impossible:
                unanswerable_total += 1

                if predicted_no_answer:
                    em = 1
                    f1 = 1
                    unanswerable_correct += 1
                else:
                    em = 0
                    f1 = 0
                    false_answer += 1

            else:
                answerable_total += 1

                if predicted_no_answer:
                    em = 0
                    f1 = 0
                    false_no_answer += 1
                else:
                    em = metric_max_over_ground_truths(
                        exact_match_score,
                        predicted_answer,
                        gold_answers
                    )

                    f1 = metric_max_over_ground_truths(
                        f1_score,
                        predicted_answer,
                        gold_answers
                    )

                    if em == 1:
                        answerable_correct += 1

            exact_match_total += em
            f1_total += f1

            evaluation_results.append({
                "id": qid,
                "question": question,
                "is_impossible": is_impossible,
                "gold_answers": gold_answers,
                "main_gold_answer": gold_answers[0] if gold_answers else "",
                "predicted_answer": predicted_answer,
                "predicted_no_answer": predicted_no_answer,
                "exact_match": em,
                "f1": f1,
                "bert_score": best_answer["score"],
                "reranked_rank": best_answer["reranked_rank"],
                "chunk_id": best_answer["chunk_id"],
                "section_title": best_answer["section_title"],
                "retrieval_score": best_answer["retrieval_score"],
                "reranker_score": best_answer["reranker_score"],
            })

exact_match = exact_match_total / total_questions
mean_f1 = f1_total / total_questions

answerable_em = answerable_correct / answerable_total if answerable_total > 0 else 0
unanswerable_accuracy = unanswerable_correct / unanswerable_total if unanswerable_total > 0 else 0

false_no_answer_rate = false_no_answer / answerable_total if answerable_total > 0 else 0
false_answer_rate = false_answer / unanswerable_total if unanswerable_total > 0 else 0

print(f"Numero totale domande: {total_questions}")
print(f"Domande answerable: {answerable_total}")
print(f"Domande unanswerable: {unanswerable_total}")
print()
print(f"Exact Match globale: {exact_match:.4f}")
print(f"F1 Score globale: {mean_f1:.4f}")
print()
print(f"Answerable EM: {answerable_em:.4f}")
print(f"No-answer accuracy: {unanswerable_accuracy:.4f}")
print()
print(f"False no-answer: {false_no_answer}")
print(f"False no-answer rate: {false_no_answer_rate:.4f}")
print()
print(f"False answer: {false_answer}")
print(f"False answer rate: {false_answer_rate:.4f}")
print()

results_df = pd.DataFrame(evaluation_results)
display(results_df.head(20))

  0%|          | 0/26 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Numero totale domande: 380
Domande answerable: 300
Domande unanswerable: 80

Exact Match globale: 0.9263
F1 Score globale: 0.9328

Answerable EM: 0.9300
No-answer accuracy: 0.9125

False no-answer: 11
False no-answer rate: 0.0367

False answer: 7
False answer rate: 0.0875



,id,question,is_impossible,gold_answers,main_gold_answer,predicted_answer,predicted_no_answer,exact_match,f1,bert_score,reranked_rank,chunk_id,section_title,retrieval_score,reranker_score
0,chunk_0001_q1,Da quali materiali è costituita la rupe del Ca...,False,[un agglomerato di tufo palagonico e di basalt...,un agglomerato di tufo palagonico e di basalto...,The answer is not present in the context.,True,0,0.0,0.929616,1,chunk_0002,LA RUPE,0.847533,0.884305
1,chunk_0001_q2,Come viene chiamata localmente la piattaforma ...,False,"[la « Praca », « Praca », Praca]",la « Praca »,Praca,False,1,1.0,0.699106,1,chunk_0002,LA RUPE,0.784193,0.990237
2,chunk_0001_q3,Da quale lato gli scisti basaltici assumono fo...,False,"[dal lato sud-ovest, lato sud-ovest, sud-ovest]",dal lato sud-ovest,sud-ovest,False,1,1.0,0.940882,1,chunk_0002,LA RUPE,0.815193,0.998223
3,chunk_0001_unanswerable_1,Quale studioso misurò per primo lo spessore de...,True,[],,The answer is not present in the context.,True,1,1.0,0.976188,1,chunk_0002,LA RUPE,0.821445,0.330889
4,chunk_0002_q1,In quale anno Croftedt presentò all'Accademia ...,False,"[l'anno 1756, 1756]",l'anno 1756,1756,False,1,1.0,1.928300,1,chunk_0002,LA RUPE,0.829840,0.997990
5,chunk_0002_q2,In quale eruzione storica il sottosuolo del Ca...,False,"[dall'eruzione storica del 1169, eruzione stor...",dall'eruzione storica del 1169,1169.,False,1,1.0,1.755936,1,chunk_0002,LA RUPE,0.823415,0.964932
6,chunk_0002_q3,Di che colore è il basalto del Castello?,False,"[ferruginoso, di colore ferruginoso]",ferruginoso,"ferruginoso,",False,1,1.0,1.864314,1,chunk_0002,LA RUPE,0.834425,0.995969
7,chunk_0002_unanswerable_1,Quale analisi chimica determinò la percentuale...,True,[],,The answer is not present in the context.,True,1,1.0,0.967810,1,chunk_0002,LA RUPE,0.848305,0.473803
8,chunk_0003_q1,Quale elemento viene ritrovato tra i prismi ba...,False,[cemento zoolitico],cemento zoolitico,cemento zoolitico,False,1,1.0,0.945305,1,chunk_0003,LA RUPE,0.819418,0.981526
9,chunk_0003_q2,Quale fenomeno viene indicato come possibile o...,False,"[un'immensa eruzione sottomarina, eruzione sot...",un'immensa eruzione sottomarina,"un'immensa eruzione sottomarina,",False,1,1.0,0.891297,1,chunk_0003,LA RUPE,0.833432,0.998298


We save the results in `.json` and `.csv` format.

In [26]:
output_path = Path("/content/drive/MyDrive/nlp_project/results/bert_qa_evaluation_results.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", encoding="utf-8") as f:
    json.dump(evaluation_results, f, ensure_ascii=False, indent=2)

csv_output_path = Path("/content/drive/MyDrive/nlp_project/results/bert_qa_evaluation_results.csv")

results_df.to_csv(csv_output_path, index=False, encoding="utf-8")

print(f"JSON results saved to: {output_path}")
print(f"CSV results saved to: {csv_output_path}")

JSON results saved to: /content/drive/MyDrive/nlp_project/results/bert_qa_evaluation_results.json
CSV results saved to: /content/drive/MyDrive/nlp_project/results/bert_qa_evaluation_results.csv


A simpler method would be to base the final selection only on the **reader score**. Therefore:
- If all produced answers are empty, the method returns “The answer is not present in the context”.
- If non-empty answers exist, the answer with the highest reader score is selected. This answer is accepted only if its score is at least equal to `min_reader_score`; otherwise, the system still returns a no-answer result.

The more complex method is implemented for three main reasons:
- to reduce false no-answer cases, namely cases in which the system says that the answer is not present even though it is present;
- to avoid selecting answers only because they have a high BERT score, while coming from poorly relevant chunks;
- to better exploit the reranker also in the final answer selection phase.

A comparison is shown below:

| Method | Global Exact Match | Global F1 Score | Answerable EM | No-answer accuracy | False no-answer | False no-answer rate | False answer | False answer rate | Wrong span | Total errors | Error percentage |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| Simple method | 0.9105 | 0.9151 | 0.9000 | **0.9500** | 19 | 0.0633 | **4** | **0.0500** | 11 | 34 | 8.95% |
| Complex method | **0.9263** | **0.9328** | **0.9300** | 0.9125 | **11** | **0.0367** | 7 | 0.0875 | **10** | **28** | **7.37%** |

The comparison shows that the complex method achieves better overall performance than the simple method. Global Exact Match increases from 0.9105 to 0.9263, while the F1 Score increases from 0.9151 to 0.9328. An improvement is also observed on answerable questions only, with Answerable EM increasing from 0.9000 to 0.9300.

The main advantage of the complex method is the reduction in false no-answer cases, which decrease from 19 to 11. This indicates that the combined use of the reader score, the reranker score, and the forcing mechanism makes it possible to recover some answers that the simple method tended to incorrectly classify as absent.

However, the complex method is less conservative on unanswerable questions. No-answer accuracy decreases from 0.9500 to 0.9125, and false answers increase from 4 to 7. This means that the system is more likely to produce an answer even in some cases where the answer is not present in the context.

Overall, the complex method is preferable because it reduces the total number of errors from 34 to 28 and improves both global Exact Match and F1. The worsening in no-answer handling is present, but limited compared to the gain obtained on answerable questions.

We perform a qualitative analysis of BERT errors:

In [27]:
def extract_chunk_id(row):
    if row["is_impossible"]:
        return None

    qid = row["id"]

    if "_q" in qid:
        return qid.rsplit("_q", 1)[0]

    return None


def classify_error_type(row):
    if row["is_impossible"] == False and row["predicted_no_answer"] == True:
        return "false_no_answer"

    if row["is_impossible"] == True and row["predicted_no_answer"] == False:
        return "false_answer"

    if row["is_impossible"] == False and row["predicted_no_answer"] == False:
        return "wrong_span"

    return "other"


errors_reader = results_df[results_df["exact_match"] == 0].copy()
errors_reader = errors_reader.sort_values("f1")

errors_reader["gold_chunk_id"] = errors_reader.apply(extract_chunk_id, axis=1)
errors_reader["error_type"] = errors_reader.apply(classify_error_type, axis=1)

print(f"Numero totale di errori del reader: {len(errors_reader)}")
print(f"Percentuale di errori: {len(errors_reader) / len(results_df) * 100:.2f}%")

false_no_answer_errors = errors_reader[
    errors_reader["error_type"] == "false_no_answer"
].copy()

false_answer_errors = errors_reader[
    errors_reader["error_type"] == "false_answer"
].copy()

wrong_span_errors = errors_reader[
    errors_reader["error_type"] == "wrong_span"
].copy()

print()
print(f"False no-answer: {len(false_no_answer_errors)}")
print(f"False answer: {len(false_answer_errors)}")
print(f"Wrong span: {len(wrong_span_errors)}")

columns_to_show = [
    "question",
    "error_type",
    "is_impossible",
    "gold_answers",
    "predicted_answer",
    "predicted_no_answer",
    "f1",
    "chunk_id",
    "gold_chunk_id",
    "section_title",
    "bert_score",
    "reranked_rank",
    "retrieval_score",
    "reranker_score"
]

display(errors_reader[columns_to_show].head(100))

output_path = Path("/content/drive/MyDrive/nlp_project/results/errors_reader.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

errors_reader[columns_to_show].to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV saved to: {output_path}")

Numero totale di errori del reader: 28
Percentuale di errori: 7.37%

False no-answer: 11
False answer: 7
Wrong span: 10


,question,error_type,is_impossible,gold_answers,predicted_answer,predicted_no_answer,f1,chunk_id,gold_chunk_id,section_title,bert_score,reranked_rank,retrieval_score,reranker_score
0,Da quali materiali è costituita la rupe del Ca...,false_no_answer,False,[un agglomerato di tufo palagonico e di basalt...,The answer is not present in the context.,True,0.000000,chunk_0002,chunk_0001,LA RUPE,0.929616,1,0.847533,0.884305
17,Quale forma si pensa che potesse avere la Rupe...,false_no_answer,False,"[appuntita a guisa di piramide, a guisa di pir...",The answer is not present in the context.,True,0.000000,chunk_0005,chunk_0005,LA RUPE,0.232796,1,0.841859,0.928751
19,Nei versi del componimento 'Castello d'Aci' qu...,false_no_answer,False,"[il maschio tuo profilo avventuroso, maschio t...",The answer is not present in the context.,True,0.000000,chunk_0092,chunk_0006,IL CANTO DEL CARCERATO,0.867561,1,0.837454,0.548558
27,Quali spostamenti vengono messi in evidenza da...,false_no_answer,False,"[dei notevoli, per quanto lenti, spostamenti n...",The answer is not present in the context.,True,0.000000,chunk_0008,chunk_0008,POCHE NOTE DI GEOLOGIA,0.112169,1,0.772057,0.686964
28,Quale tipo di rocce costituisce il cono dell’E...,false_no_answer,False,"[materiali basaltici e doloritici, basaltici e...",The answer is not present in the context.,True,0.000000,chunk_0008,chunk_0008,POCHE NOTE DI GEOLOGIA,0.234963,1,0.839116,0.437767
32,"In quale piana geologica, situata a meridione ...",false_no_answer,False,"[piana neozoica, neozoica]",The answer is not present in the context.,True,0.000000,chunk_0009,chunk_0009,POCHE NOTE DI GEOLOGIA,0.959257,1,0.836372,0.977805
107,Chi fece un voto alla Madonna promettendo di e...,wrong_span,False,"[il Conte Ruggiero, Conte Ruggiero, Ruggiero]",Dionisio,False,0.000000,chunk_0034,chunk_0030,VALVERDE,0.428061,2,0.841914,0.939775
97,Quale costruzione saracena riportava il motto ...,false_answer,True,[],ruderi di una costruzione,False,0.000000,chunk_0027,None,PERIODO NORMANNO,0.014069,1,0.814727,0.921176
206,"Chi tenne la giovane Maria, erede al trono di ...",wrong_span,False,[Artale],non intendeva sposarla.,False,0.000000,chunk_0057,chunk_0057,GLI ALAGONA,0.021728,1,0.871361,0.999977
213,Quale emissario portò a Palermo la promessa di...,false_answer,True,[],"Manfredi,",False,0.000000,chunk_0058,None,RE MARTINO,0.863862,1,0.821646,0.991355


CSV saved to: /content/drive/MyDrive/nlp_project/results/errors_reader.csv


# **Comparison between Read Comprehension models**

In this phase, the impact of the **extractive read comprehension** model on the final quality of the Question Answering pipeline is evaluated. The FAISS retrieval component and the reranking module are kept unchanged, while only the reader used to extract the answer from the selected chunks is replaced.

The objective of the experiment is to understand how much the choice of the reader affects the final performance of the system. Given the same question and the same retrieved contexts, each model receives the same reranked chunks as input and must identify the text span that represents the most likely answer.

Five models were compared:

- `deepset/xlm-roberta-base-squad2`: multilingual model based on XLM-RoBERTa, used as the main baseline of the pipeline;
- `deepset/xlm-roberta-large-squad2`: larger version of the previous model, introduced to verify whether greater reader capacity improves correct span extraction;
- `anakin87/electra-italian-xxl-cased-squad-it`: Italian-specific model based on ELECTRA and fine-tuned on an Italian dataset for Question Answering;
- `mrm8488/bert-italian-finedtuned-squadv1-it-alfa`: Italian BERT model fine-tuned for the extractive Question Answering task;
- `mrm8488/umberto-wikipedia-uncased-v1-finetuned-squadv1-it`: model based on UmBERTo, specific to Italian texts and adapted to the extractive QA task.

For each reader, the same metrics are computed: **Exact Match**, **F1 Score**, number of errors, error rate, average reader score, average rank of the chunk from which the answer is extracted, and average inference time per question.

The average time per question is computed by dividing the total inference time by the number of answerable questions evaluated. For each question, time is measured from the beginning of the retrieval phase to the selection of the final answer. Consequently, it includes FAISS retrieval, chunk reranking, and the application of the reader to the candidate contexts.

In this way, it is possible to compare not only the accuracy of the different models, but also their computational cost and their ability to correctly exploit the chunks provided by the reranker. Since retrieval and reranking remain unchanged, the differences observed in the metrics mainly depend on the ability of each reader to identify the correct span within the context.

In [28]:
reader_models = [
    "deepset/xlm-roberta-base-squad2",
    "deepset/xlm-roberta-large-squad2",
    "anakin87/electra-italian-xxl-cased-squad-it",
    "mrm8488/bert-italian-finedtuned-squadv1-it-alfa",
    "mrm8488/umberto-wikipedia-uncased-v1-finetuned-squadv1-it"
]

qa_path = Path("/content/drive/MyDrive/nlp_project/data/qa_dataset_squad_like_final.json")
data = json.loads(qa_path.read_text(encoding="utf-8"))

def evaluate_reader_model(model_name, faiss_top_k=20, reader_top_k=5):
    device = 0 if torch.cuda.is_available() else -1

    qa_reader = pipeline(
        "question-answering",
        model=model_name,
        tokenizer=model_name,
        device=device
    )

    evaluation_results = []

    total_questions = 0
    exact_match_total = 0
    f1_total = 0
    total_time = 0

    for section in tqdm(data["data"], desc=f"Valutazione {model_name}"):
        for paragraph in section["paragraphs"]:
            for qa in paragraph["qas"]:
                if qa.get("is_impossible", False):
                    continue

                question = qa["question"]
                gold_answers = [answer["text"] for answer in qa["answers"]]
                qid = qa["id"]

                start_time = time.time()

                retrieved_chunks = retrieve_faiss_reranked(
                    question,
                    faiss_top_k=faiss_top_k,
                    rerank_top_k=reader_top_k
                )

                candidate_answers = []

                for rank, chunk in enumerate(retrieved_chunks, start=1):
                    result = qa_reader(
                        question=question,
                        context=chunk["context"]
                    )

                    candidate_answers.append({
                        "answer": result["answer"],
                        "score": float(result["score"]),
                        "start": int(result["start"]),
                        "end": int(result["end"]),
                        "reranked_rank": rank,
                        "chunk_id": chunk["chunk_id"],
                        "section_title": chunk["section_title"],
                        "retrieval_score": chunk["score"],
                        "reranker_score": chunk["reranker_score"]
                    })

                best_answer = max(candidate_answers, key=lambda x: x["score"])
                predicted_answer = best_answer["answer"]

                elapsed_time = time.time() - start_time
                total_time += elapsed_time

                em = metric_max_over_ground_truths(
                    exact_match_score,
                    predicted_answer,
                    gold_answers
                )

                f1 = metric_max_over_ground_truths(
                    f1_score,
                    predicted_answer,
                    gold_answers
                )

                total_questions += 1
                exact_match_total += em
                f1_total += f1

                evaluation_results.append({
                    "model_name": model_name,
                    "id": qid,
                    "question": question,
                    "gold_answers": gold_answers,
                    "predicted_answer": predicted_answer,
                    "exact_match": em,
                    "f1": f1,
                    "bert_score": best_answer["score"],
                    "reranked_rank": best_answer["reranked_rank"],
                    "chunk_id": best_answer["chunk_id"],
                    "section_title": best_answer["section_title"],
                    "retrieval_score": best_answer["retrieval_score"],
                    "reranker_score": best_answer["reranker_score"],
                    "time_seconds": elapsed_time
                })

    results_df_model = pd.DataFrame(evaluation_results)

    summary = {
        "model_name": model_name,
        "num_answerable_questions": total_questions,
        "exact_match": exact_match_total / total_questions,
        "f1": f1_total / total_questions,
        "errors": int((results_df_model["exact_match"] == 0).sum()),
        "error_rate": float((results_df_model["exact_match"] == 0).mean()),
        "avg_reader_score": float(results_df_model["bert_score"].mean()),
        "avg_reranked_rank": float(results_df_model["reranked_rank"].mean()),
        "avg_time_per_question": total_time / total_questions,
        "total_time_seconds": total_time
    }

    return summary, results_df_model

In [29]:
comparison_results = []
all_reader_results = []

for model_name in reader_models:
    print("=" * 120)
    print(f"Evaluating model: {model_name}")
    print("=" * 120)

    summary, model_results_df = evaluate_reader_model(
        model_name=model_name,
        faiss_top_k=20,
        reader_top_k=5
    )

    comparison_results.append(summary)
    all_reader_results.append(model_results_df)

    output_path = Path(f"/content/drive/MyDrive/nlp_project/results/reader_eval_{model_name.replace('/', '_')}.csv")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    model_results_df.to_csv(output_path, index=False, encoding="utf-8")

    print(summary)

    del model_results_df

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Evaluating model: deepset/xlm-roberta-base-squad2


config.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaForQuestionAnswering LOAD REPORT from: deepset/xlm-roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/801k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

Valutazione mrm8488/umberto-wikipedia-uncased-v1-finetuned-squadv1-it:   0%|          | 0/26 [00:00<?, ?it/s]

{'model_name': 'mrm8488/umberto-wikipedia-uncased-v1-finetuned-squadv1-it', 'num_answerable_questions': 300, 'exact_match': 0.07333333333333333, 'f1': 0.10578602878602879, 'errors': 278, 'error_rate': 0.9266666666666666, 'avg_reader_score': 0.8523119324476203, 'avg_reranked_rank': 2.6, 'avg_time_per_question': 0.7550552598635356, 'total_time_seconds': 226.51657795906067}


In [30]:
comparison_df = pd.DataFrame(comparison_results)

comparison_df = comparison_df.sort_values(
    by=["f1", "exact_match"],
    ascending=False
)

display(comparison_df)

comparison_output_path = Path("/content/drive/MyDrive/nlp_project/results/readers_comparison.csv")
comparison_df.to_csv(comparison_output_path, index=False, encoding="utf-8")

print(f"Comparison table saved to: {comparison_output_path}")

,model_name,num_answerable_questions,exact_match,f1,errors,error_rate,avg_reader_score,avg_reranked_rank,avg_time_per_question,total_time_seconds
1,deepset/xlm-roberta-large-squad2,300,0.930000,0.940910,21,0.070000,0.985704,1.370000,1.144704,343.411150
0,deepset/xlm-roberta-base-squad2,300,0.846667,0.871784,46,0.153333,0.741012,1.413333,0.520231,156.069238
2,anakin87/electra-italian-xxl-cased-squad-it,300,0.673333,0.713608,98,0.326667,0.956402,1.753333,0.484009,145.202679
3,mrm8488/bert-italian-finedtuned-squadv1-it-alfa,300,0.513333,0.565159,146,0.486667,1.129689,2.190000,0.482812,144.843458
4,mrm8488/umberto-wikipedia-uncased-v1-finetuned...,300,0.073333,0.105786,278,0.926667,0.852312,2.600000,0.755055,226.516578


Comparison table saved to: /content/drive/MyDrive/nlp_project/results/readers_comparison.csv


In [31]:
plot_df = comparison_df.copy()

plot_df["model_short"] = plot_df["model_name"].apply(lambda x: x.split("/")[-1])

metrics_df = plot_df.melt(
    id_vars="model_short",
    value_vars=["exact_match", "f1"],
    var_name="metric",
    value_name="score"
)

fig = px.bar(
    metrics_df,
    x="model_short",
    y="score",
    color="metric",
    barmode="group",
    text=metrics_df["score"].round(4),
    title="Comparison between reader models: Exact Match and F1"
)

fig.update_layout(
    xaxis_title="Reader model",
    yaxis_title="Score",
    yaxis=dict(range=[0, 1]),
    width=1200,
    height=600,
    xaxis_tickangle=-30
)

fig.update_traces(
    textposition="outside"
)

fig.show()

In [32]:
fig = px.bar(
    plot_df,
    x="model_short",
    y="avg_time_per_question",
    text=plot_df["avg_time_per_question"].round(3),
    title="Average time per question of reader models"
)

fig.update_layout(
    xaxis_title="Reader model",
    yaxis_title="Seconds per question",
    width=1200,
    height=600,
    xaxis_tickangle=-30
)

fig.update_traces(
    textposition="outside"
)

fig.show()

The comparison between the different reader models shows that `deepset/xlm-roberta-large-squad2` achieves the best overall performance, with an Exact Match of 0.93 and an F1 Score of 0.94. Compared to the base version, the large model reduces the number of errors from 46 to 21, showing a significant improvement in its ability to identify the correct span within the reranked chunks.

The base version of XLM-RoBERTa still represents **a good trade-off between accuracy and computational cost**, since it maintains high performance with a lower average time per question. By contrast, the Italian-specific models do not outperform it in the considered domain: Italian ELECTRA obtains intermediate results, while the Italian BERT and UmBERTo models show substantially lower performance.

These results indicate that, for the developed extractive RAG system, the robustness of the read comprehension model in the question answering task is more important than language-specific specialization on Italian alone. Therefore, the `deepset/xlm-roberta-large-squad2` model is selected as the final reader of the pipeline.